In [1]:
# ==================================================================================================
# PROJECT 24 — STEP 5A PARALLEL WORKER
# RESUME-SAFE CHECKPOINT-SCHEMA-COMPATIBLE ACCELERATED SEED-SHARD EXECUTION
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS IN ITS ASSIGNED NEW PARALLEL COLAB WORKER NOTEBOOK ONLY.
# DO NOT RUN THIS WORKER CELL IN THE MASTER NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 24 Step 4B smoke-test checkpoint and every upstream contract;
# - validate the accelerated REC engine against the exact frozen Step 4B smoke outputs;
# - execute only this worker's assigned non-overlapping seed shard;
# - checkpoint each completed condition independently using atomic output files;
# - resume safely after a Colab disconnect by skipping only fully validated conditions;
# - fit the four frozen ML techniques and evaluate the three frozen baselines;
# - write ranked-test, build-metric, project-run, model-fit, median, and audit outputs;
# - freeze an independent worker checkpoint; the master Step 5A freeze happens only after all workers PASS.
#
# SAFETY:
# - no registry write;
# - no modification of Projects 1–23;
# - six total seed shards are executed in two waves of three Colab runtimes;
# - no prior-project condition-output access;
# - clean evaluation data remain immutable;
# - incomplete assigned-condition outputs are preserved in this worker's private quarantine before rerun.
# - master Step 5A aggregate/checkpoint/status files are write-protected by an explicit guard.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import gc
import hashlib
import json
import os
import platform
import shutil
import time
import warnings
import tarfile

from google.colab import drive

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)

print("=" * 136)
print("=== PROJECT 24 STEP 5A PARALLEL WORKER: SEEDS 01-05 ===")
print("=" * 136)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)
EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_24_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)
STEP5A_STATUS = (
    "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
)
CONDITION_STATUS = "PASS_FULL_CONDITION"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "ff9ed031d21d67bf218d5eab70bd8502ca39953adedbe840348f382d5599314e"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "3f5a0cc24260b251fd8cd43f7391ff7d3cc3d517e8f79a8ad443f4002cdc9ff2"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)
EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)
EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)
EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_REC_IMPLEMENTATION = (
    "PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_"
    "ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT"
)

EXPECTED_STEP3A_CODE_REVISION = (
    "PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER_SCHEMA_FIX"
)

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

# Worker-shard constants are injected into each generated worker file.
WORKER_SEED_START = 1
WORKER_SEED_END = 5
WORKER_TAG = "seed_01_05"
WORKER_NOTEBOOK_NAME = "Thesis_p24_seed1-5"
WORKER_CODE_REVISION = "P24_SEEDS_01_05_V1_FROZEN_SCHEMA_SMOKE_EQUIVALENT"
WORKER_STATUS = "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_01_05_COMPLETE"
WORKER_IS_SMOKE_OWNER = (WORKER_SEED_START <= 1 <= WORKER_SEED_END)
SHARED_EQUIVALENCE_WAIT_SECONDS = 60 * 60
SHARED_EQUIVALENCE_POLL_SECONDS = 10

EXPECTED_REGISTERED_PROJECTS = 23

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_ROWS = 224_550
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_FAILING_EVAL_BUILDS = 17
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 1_072
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = EXPECTED_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_RNG_ROWS = 121_224_030
ACCELERATED_ENGINE_VERSION = "PROJECT_24_FAST_DEPENDENT_REC_V1_SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE"
SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
WORKER_REPETITION_SEEDS = list(range(WORKER_SEED_START, WORKER_SEED_END + 1))
EXPECTED_WORKER_CONDITIONS = len(WORKER_REPETITION_SEEDS) * len(NOISE_LEVELS)
EXPECTED_WORKER_RAW_FILES = EXPECTED_WORKER_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_WORKER_RANKING_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_RANKING_ROWS_PER_CONDITION
EXPECTED_WORKER_BUILD_METRIC_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
EXPECTED_WORKER_PROJECT_RUN_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
EXPECTED_WORKER_MODEL_FITS = EXPECTED_WORKER_CONDITIONS * EXPECTED_MODEL_FIT_ROWS_PER_CONDITION
EXPECTED_WORKER_TRAINING_MEDIAN_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SOURCE_DIR = Path("/content/datasets/datasets/SonarSource@sonarqube")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_24_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_24_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_24_fixed_chronological_builds.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_24_selection_checkpoint.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
BUILD_ENTITY_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
CLEAN_RECONSTRUCTED_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
CLEAN_ANCHOR_OFFSETS_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_24_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
MODEL_RAW_TRAIN_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
MODEL_RAW_EVAL_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_24_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
PREDICTOR_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_predictor_contract.csv"
MODEL_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_model_contract.json"
BASELINE_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_baseline_contract.json"
RANKING_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_ranking_contract.json"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_24_runtime_contract_checkpoint.json"

STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_24_smoke_test_checkpoint.json"
SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"

# Official master Step 5A outputs. Workers may READ their state for guards but MUST NOT write them.
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
MASTER_RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_24_step5a_checkpoint.json"

# Shared accelerated-equivalence audit: ONLY the seed-1 worker writes it; other workers wait/read it.
ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT
    / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

# Private worker outputs: no path overlaps another worker.
PARALLEL_WORKERS_ROOT = FULL_EXPERIMENT_ROOT / "parallel_workers"
WORKER_ROOT = PARALLEL_WORKERS_ROOT / WORKER_TAG
INCOMPLETE_BACKUP_ROOT = WORKER_ROOT / "incomplete_condition_backups"
WORKER_INVENTORY_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_condition_inventory.csv"
WORKER_RAW_MANIFEST_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_raw_manifest.csv"
WORKER_BASELINE_INVARIANCE_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_baseline_invariance.csv"
WORKER_COMBINED_AUDIT_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_combined_condition_audit.csv"
WORKER_VALIDATION_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_validation.csv"
WORKER_REPORT_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_report.json"
RUN_PROGRESS_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_run_progress.json"
WORKER_STATUS_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_status.json"
WORKER_CHECKPOINT_PATH = NOTES_ROOT / f"project_24_step5a_worker_{WORKER_TAG}_checkpoint.json"

# --------------------------------------------------------------------------------------------------
# 2B. FRESH-WORKER RUNTIME BOOTSTRAP
# --------------------------------------------------------------------------------------------------

print("Worker notebook:", WORKER_NOTEBOOK_NAME)
print("Assigned repetition seeds:", WORKER_REPETITION_SEEDS)
print("Assigned conditions:", EXPECTED_WORKER_CONDITIONS)
print("Mounting the shared thesis Google Drive.")

drive.mount(
    "/content/drive",
    force_remount=False,
)

ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
REQUIRED_SOURCE_FILE_NAMES = {
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
}

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(
        "Frozen TCP-CI archive is missing from Google Drive:\n"
        f"{ARCHIVE_PATH}"
    )

source_ready = bool(
    SOURCE_DIR.is_dir()
    and REQUIRED_SOURCE_FILE_NAMES.issubset({
        path.name
        for path in SOURCE_DIR.iterdir()
        if path.is_file()
    })
)

if not source_ready:
    print("Restoring only SonarSource@sonarqube from the frozen TCP-CI archive into this worker runtime.")
    local_dataset_root = Path("/content/datasets")
    local_dataset_root.mkdir(parents=True, exist_ok=True)
    member_prefix = "datasets/SonarSource@sonarqube/"
    extracted_files = 0
    resolved_root = local_dataset_root.resolve()

    with tarfile.open(ARCHIVE_PATH, mode="r:gz") as archive:
        for member in archive:
            member_name = member.name.replace("\\", "/").lstrip("/")
            if not (
                member_name == "datasets/SonarSource@sonarqube"
                or member_name.startswith(member_prefix)
            ):
                continue

            target_path = local_dataset_root / member_name
            resolved_target = target_path.resolve()
            if resolved_target != resolved_root and resolved_root not in resolved_target.parents:
                raise RuntimeError(
                    "Unsafe archive member encountered during worker bootstrap:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                target_path.parent.mkdir(parents=True, exist_ok=True)
                source_handle = archive.extractfile(member)
                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )
                with source_handle, target_path.open("wb") as output_handle:
                    shutil.copyfileobj(source_handle, output_handle, length=8 * 1024 * 1024)
                extracted_files += 1

    print("Project source files restored:", extracted_files)

source_file_names = {
    path.name
    for path in SOURCE_DIR.iterdir()
    if path.is_file()
} if SOURCE_DIR.is_dir() else set()
missing_worker_source_files = sorted(REQUIRED_SOURCE_FILE_NAMES - source_file_names)
if missing_worker_source_files:
    raise FileNotFoundError(
        "Worker runtime could not restore the complete frozen SonarSource@sonarqube source:\n"
        + "\n".join(missing_worker_source_files)
    )

WORKER_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)

def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)

def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)

def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()

def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")

def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })

def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )

def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )

def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )

def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )

def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )

def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)

def reconstruct_dependent_rec_fast(condition_combined_verdict):
    """
    Reconstruct only the 13 verdict-dependent REC features.

    This is algebraically equivalent to the frozen Step 4B implementation:
    - the exact Step 2B-frozen InferredTestOrder is used;
    - only prior executions contribute to each current row;
    - recent window = 6;
    - verdict 2 = assertion, verdict 1 = exception;
    - file-history rates use distinct prior target builds and current-build entities;
    - builds with no mapped entities produce 0 when target history exists and -1 when it does not.
    """
    condition_combined_verdict = np.asarray(
        condition_combined_verdict,
        dtype=np.int16,
    )

    if len(condition_combined_verdict) != EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS:
        raise RuntimeError(
            "Accelerated REC engine received the wrong execution-history length."
        )

    verdict_sorted = condition_combined_verdict[
        accelerated_history_combined_indices
    ]

    result = np.full(
        (
            EXPECTED_MODEL_ROWS,
            len(VERDICT_DEPENDENT_REC),
        ),
        -1.0,
        dtype=np.float64,
    )

    for group_index in range(accelerated_group_count):
        requested_model_indices = accelerated_requested_model_indices[group_index]

        if len(requested_model_indices) == 0:
            continue

        start = int(accelerated_group_starts[group_index])
        end = int(accelerated_group_ends[group_index])
        local_positions = accelerated_requested_local_positions[group_index]

        verdict = verdict_sorted[start:end]
        group_length = len(verdict)
        position = np.arange(group_length, dtype=np.int64)

        failure = verdict > 0
        assertion = verdict == 2
        exception = verdict == 1
        transition = np.zeros(group_length, dtype=np.bool_)

        if group_length > 1:
            transition[1:] = verdict[1:] != verdict[:-1]

        failure_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(failure, dtype=np.int64),
        ))
        assertion_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(assertion, dtype=np.int64),
        ))
        exception_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(exception, dtype=np.int64),
        ))
        transition_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(transition, dtype=np.int64),
        ))

        has_history = local_positions > 0

        if has_history.any():
            requested_with_history = np.flatnonzero(has_history)
            current_positions = local_positions[requested_with_history]
            model_indices = requested_model_indices[requested_with_history]

            recent_starts = np.maximum(
                0,
                current_positions - RECENT_WINDOW,
            )
            recent_lengths = current_positions - recent_starts

            last_failure_position = np.maximum.accumulate(
                np.where(failure, position, -1)
            )
            last_transition_position = np.maximum.accumulate(
                np.where(transition, position, -1)
            )

            prior_last_failure = last_failure_position[
                current_positions - 1
            ]
            prior_last_transition = last_transition_position[
                current_positions - 1
            ]

            result[model_indices, 0] = np.where(
                prior_last_failure >= 0,
                current_positions - 1 - prior_last_failure,
                -1,
            ).astype(np.float64)
            result[model_indices, 1] = np.where(
                prior_last_transition >= 0,
                current_positions - 1 - prior_last_transition,
                -1,
            ).astype(np.float64)

            result[model_indices, 2] = (
                failure_prefix[current_positions]
                - failure_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 3] = (
                assertion_prefix[current_positions]
                - assertion_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 4] = (
                exception_prefix[current_positions]
                - exception_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 5] = (
                transition_prefix[current_positions]
                - transition_prefix[recent_starts]
            ) / recent_lengths

            result[model_indices, 6] = (
                failure_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 7] = (
                assertion_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 8] = (
                exception_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 9] = (
                transition_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 10] = verdict[
                current_positions - 1
            ].astype(np.float64)

        # File-history features. The counters contain only target executions
        # strictly before the current position, matching Step 4B exactly.
        failure_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        transition_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        failure_denominator = 0
        transition_denominator = 0
        requested_pointer = 0

        group_build_indices = accelerated_history_build_dense_indices[
            start:end
        ]

        for local_position in range(group_length):
            while (
                requested_pointer < len(local_positions)
                and int(local_positions[requested_pointer]) == local_position
            ):
                model_index = int(
                    requested_model_indices[requested_pointer]
                )

                if local_position > 0:
                    current_entities = accelerated_build_entity_arrays[
                        int(group_build_indices[local_position])
                    ]

                    if failure_denominator == 0:
                        result[model_index, 11] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 11] = 0.0
                    else:
                        result[model_index, 11] = float(
                            failure_entity_counts[
                                current_entities
                            ].max()
                            / failure_denominator
                        )

                    if transition_denominator == 0:
                        result[model_index, 12] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 12] = 0.0
                    else:
                        result[model_index, 12] = float(
                            transition_entity_counts[
                                current_entities
                            ].max()
                            / transition_denominator
                        )

                requested_pointer += 1

            changed_entities = accelerated_build_entity_arrays[
                int(group_build_indices[local_position])
            ]

            if failure[local_position]:
                if len(changed_entities) != 0:
                    failure_entity_counts[changed_entities] += 1
                failure_denominator += 1

            if transition[local_position]:
                if len(changed_entities) != 0:
                    transition_entity_counts[changed_entities] += 1
                transition_denominator += 1

        if requested_pointer != len(local_positions):
            raise RuntimeError(
                "Accelerated REC engine did not emit every requested row."
            )

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Accelerated REC engine produced non-finite values."
        )

    return result

def maximum_absolute_difference(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)

    if left.shape != right.shape:
        return np.inf

    if left.size == 0:
        return 0.0

    return float(np.max(np.abs(left - right)))

def compare_full_condition_to_smoke(condition_key, full_condition_dir):
    """Compare all scientific outputs with the frozen Step 4B condition."""
    full_condition_dir = Path(full_condition_dir)
    smoke_condition_dir = SMOKE_ROOT / condition_key

    required_names = [
        "rankings.csv.gz",
        "build_metrics.csv",
        "project_runs.csv",
        "model_fits.csv",
        "training_medians.csv",
        "condition_audit.csv",
    ]

    for name in required_names:
        if not (full_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Accelerated equivalence input missing: {full_condition_dir / name}"
            )
        if not (smoke_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Frozen smoke output missing: {smoke_condition_dir / name}"
            )

    actual_rankings = pd.read_csv(
        full_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_rankings = pd.read_csv(
        smoke_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)

    ranking_key_columns = [
        "Technique",
        "Build",
        "Test",
        "Rank",
        "CleanVerdict",
        "CleanFailure",
    ]
    ranking_keys_equal = bool(
        len(actual_rankings) == len(smoke_rankings)
        and actual_rankings[ranking_key_columns].equals(
            smoke_rankings[ranking_key_columns]
        )
    )
    ranking_score_max_difference = maximum_absolute_difference(
        actual_rankings["Score"].to_numpy(dtype=float),
        smoke_rankings["Score"].to_numpy(dtype=float),
    )

    actual_build = pd.read_csv(
        full_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_build = pd.read_csv(
        smoke_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    build_keys_equal = bool(
        len(actual_build) == len(smoke_build)
        and actual_build[["Technique", "Build", "Tests", "Failures"]].equals(
            smoke_build[["Technique", "Build", "Tests", "Failures"]]
        )
    )
    build_metric_max_difference = maximum_absolute_difference(
        actual_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
        smoke_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
    )

    actual_project = pd.read_csv(
        full_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_project = pd.read_csv(
        smoke_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    project_keys_equal = bool(
        len(actual_project) == len(smoke_project)
        and actual_project[[
            "Technique",
            "EvaluationBuilds",
            "ScoredFailingBuilds",
            "EvaluationRows",
            "EvaluationFailures",
        ]].equals(
            smoke_project[[
                "Technique",
                "EvaluationBuilds",
                "ScoredFailingBuilds",
                "EvaluationRows",
                "EvaluationFailures",
            ]]
        )
    )
    project_metric_max_difference = maximum_absolute_difference(
        actual_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
        smoke_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
    )

    actual_medians = pd.read_csv(
        full_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    smoke_medians = pd.read_csv(
        smoke_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    median_keys_equal = bool(
        len(actual_medians) == len(smoke_medians)
        and actual_medians[["PredictorOrder", "Predictor"]].equals(
            smoke_medians[["PredictorOrder", "Predictor"]]
        )
    )
    median_max_difference = maximum_absolute_difference(
        actual_medians["TrainingMedian"].to_numpy(dtype=float),
        smoke_medians["TrainingMedian"].to_numpy(dtype=float),
    )

    actual_fits = pd.read_csv(
        full_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_fits = pd.read_csv(
        smoke_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    fit_contract_columns = [
        "Technique",
        "TrainingRows",
        "TrainingFailures",
        "Predictors",
        "ClassesJSON",
        "Status",
        "Error",
    ]
    fit_contract_equal = bool(
        len(actual_fits) == len(smoke_fits)
        and actual_fits[fit_contract_columns].equals(
            smoke_fits[fit_contract_columns]
        )
    )

    actual_audit = pd.read_csv(
        full_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    smoke_audit = pd.read_csv(
        smoke_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    # Project 24 Step 4B froze the worker-compatible condition-audit schema used here.
    # The Project 23 worker precedent used a larger audit schema, so comparing
    # those predecessor-only metadata columns against Project 24's smoke output
    # raises KeyError (for example RawTrainingRows).
    #
    # Smoke equivalence must compare the scientific audit fields that were
    # actually frozen by Project 24 Step 4B. Extra full-condition metadata are
    # validated separately by the worker's own condition-completion checks.
    audit_columns = [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "ConditionKey",
        "NoisePercent",
        "RepetitionSeed",
        "NumberFlipped",
        "ModelLabelChanges",
        "TrainingFailures",
        "DependentRECChanges",
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
        "RankingRows",
        "BuildMetricRows",
        "ProjectRunRows",
        "TrainingMedianRows",
    ]

    missing_actual_audit_columns = [
        column
        for column in audit_columns
        if column not in actual_audit.index
    ]
    missing_smoke_audit_columns = [
        column
        for column in audit_columns
        if column not in smoke_audit.index
    ]

    if missing_actual_audit_columns or missing_smoke_audit_columns:
        raise RuntimeError(
            "Project 24 smoke-equivalence audit schema differs. "
            f"Missing from full condition={missing_actual_audit_columns}; "
            f"missing from frozen smoke={missing_smoke_audit_columns}; "
            f"full columns={actual_audit.index.tolist()}; "
            f"smoke columns={smoke_audit.index.tolist()}"
        )

    audit_equal = bool(
        all(
            str(actual_audit[column]) == str(smoke_audit[column])
            for column in audit_columns
        )
    )

    passed = bool(
        ranking_keys_equal
        and ranking_score_max_difference <= 1e-12
        and build_keys_equal
        and build_metric_max_difference <= 1e-12
        and project_keys_equal
        and project_metric_max_difference <= 1e-12
        and median_keys_equal
        and median_max_difference <= 1e-12
        and fit_contract_equal
        and audit_equal
    )

    return {
        "ConditionKey": condition_key,
        "EngineVersion": ACCELERATED_ENGINE_VERSION,
        "RankingKeysEqual": ranking_keys_equal,
        "RankingScoreMaxDifference": ranking_score_max_difference,
        "BuildMetricKeysEqual": build_keys_equal,
        "BuildMetricMaxDifference": build_metric_max_difference,
        "ProjectRunKeysEqual": project_keys_equal,
        "ProjectMetricMaxDifference": project_metric_max_difference,
        "TrainingMedianKeysEqual": median_keys_equal,
        "TrainingMedianMaxDifference": median_max_difference,
        "ModelFitContractEqual": fit_contract_equal,
        "ConditionAuditEqual": audit_equal,
        "Pass": passed,
    }

def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)

def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]

def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs

DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]

def directory_manifest(root):
    root = Path(root)
    rows = []

    if root.exists():
        for path in sorted(
            [
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ],
            key=lambda candidate: candidate.relative_to(root).as_posix(),
        ):
            rows.append({
                "RelativePath": path.relative_to(root).as_posix(),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            })

    return pd.DataFrame(
        rows,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )

def directory_root_hash(manifest):
    if manifest is None:
        raise TypeError(
            "Directory manifest cannot be None."
        )

    missing_columns = [
        column
        for column in DIRECTORY_MANIFEST_COLUMNS
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Directory manifest is missing required columns: "
            + ", ".join(missing_columns)
        )

    digest = hashlib.sha256()

    if manifest.empty:
        return digest.hexdigest()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()

def path_guard_state(path):
    path = Path(path)
    if not path.exists():
        return {"Exists": False, "Bytes": 0, "SHA256": ""}
    if not path.is_file():
        raise RuntimeError(f"Master write-guard path is unexpectedly not a file: {path}")
    return {
        "Exists": True,
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }


def load_valid_shared_equivalence():
    if not ACCELERATED_EQUIVALENCE_PATH.is_file():
        return None
    try:
        frame = pd.read_csv(ACCELERATED_EQUIVALENCE_PATH, low_memory=False)
    except Exception:
        return None
    required_columns = {"ConditionKey", "Pass"}
    if not required_columns.issubset(frame.columns):
        return None
    if len(frame) != len(SMOKE_EQUIVALENCE_KEYS):
        return None
    if sorted(frame["ConditionKey"].astype(str).tolist()) != sorted(SMOKE_EQUIVALENCE_KEYS):
        return None
    pass_values = frame["Pass"]
    if pass_values.dtype == bool:
        passed = pass_values
    else:
        passed = pass_values.astype(str).str.strip().str.lower().map({"true": True, "false": False})
        if passed.isna().any():
            return None
    if not bool(passed.all()):
        return None
    return frame


def wait_for_shared_equivalence():
    deadline = time.time() + SHARED_EQUIVALENCE_WAIT_SECONDS
    announced = False
    while time.time() < deadline:
        frame = load_valid_shared_equivalence()
        if frame is not None:
            return frame
        if not announced:
            print(
                "Waiting for seed-1 worker to freeze the two accelerated smoke-equivalence checks..."
            )
            announced = True
        time.sleep(SHARED_EQUIVALENCE_POLL_SECONDS)
    raise TimeoutError(
        "Timed out waiting for the seed-1 worker accelerated-equivalence audit. "
        "Check Worker A output before retrying this worker cell."
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH,
    SMOKE_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 24 Step 5A inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint_sha256 = sha256_file(
    SMOKE_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 runtime-contract checkpoint SHA-256 differs."
    )

if smoke_checkpoint_sha256 != EXPECTED_SMOKE_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 smoke-test checkpoint SHA-256 differs."
    )

runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)
step4b_status = load_json(
    STEP4B_STATUS_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )

if step4b_status.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 24 Step 4B status is not frozen successfully."
    )

if smoke_checkpoint.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 24 smoke-test checkpoint is not frozen successfully."
    )

if smoke_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Smoke-test checkpoint project identity differs."
    )

if smoke_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Smoke-test checkpoint project slug differs."
    )

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

if rec_checkpoint.get("ImplementationVersion") != EXPECTED_REC_IMPLEMENTATION:
    raise RuntimeError(
        "Frozen Project 24 REC implementation differs."
    )

if noise_plan_checkpoint.get("Step3ACodeRevision") != EXPECTED_STEP3A_CODE_REVISION:
    raise RuntimeError(
        "Frozen Project 24 Step 3A code revision differs."
    )

if runtime_checkpoint.get("Step3ACodeRevision") != EXPECTED_STEP3A_CODE_REVISION:
    raise RuntimeError(
        "Runtime checkpoint Step 3A revision linkage differs."
    )

if smoke_checkpoint.get("RECImplementation") != EXPECTED_REC_IMPLEMENTATION:
    raise RuntimeError(
        "Smoke checkpoint REC implementation linkage differs."
    )

if smoke_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Smoke-test checkpoint active reservations differ."
    )

# Step 4B validates the runtime-priority rule against the frozen Step 4A
# runtime checkpoint, but its checkpoint schema does not duplicate that field.
# Therefore, validate the frozen linkage instead of requiring an absent key.
if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke-test checkpoint does not link to the frozen runtime contract."
    )

if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError(
        "Smoke-test checkpoint does not authorise the full experiment."
    )

smoke_output_manifest = smoke_checkpoint.get("OutputManifest", [])
if not isinstance(smoke_output_manifest, list) or not smoke_output_manifest:
    raise RuntimeError(
        "Smoke-test checkpoint does not contain an output manifest."
    )

smoke_output_manifest_failures = 0
for item in smoke_output_manifest:
    output_path = Path(item["Path"])
    if (
        not output_path.is_file()
        or int(output_path.stat().st_size) != int(item["Bytes"])
        or sha256_file(output_path) != str(item["SHA256"])
    ):
        smoke_output_manifest_failures += 1

if smoke_output_manifest_failures != 0:
    raise RuntimeError(
        "One or more frozen Step 4B smoke outputs changed."
    )

# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

actual_runtime_versions = {
    "Python": platform.python_version(),
    "numpy": metadata.version("numpy"),
    "pandas": metadata.version("pandas"),
    "scikit-learn": metadata.version("scikit-learn"),
    "xgboost": metadata.version("xgboost"),
    "lightgbm": metadata.version("lightgbm"),
    "pyarrow": metadata.version("pyarrow"),
}

runtime_version_mismatches = {
    key: {
        "Expected": EXPECTED_RUNTIME_VERSIONS[key],
        "Actual": actual_runtime_versions[key],
    }
    for key in EXPECTED_RUNTIME_VERSIONS
    if actual_runtime_versions[key] != EXPECTED_RUNTIME_VERSIONS[key]
}

if runtime_version_mismatches:
    raise RuntimeError(
        "Fresh worker runtime does not match the frozen Step 4A environment:\n"
        + json.dumps(runtime_version_mismatches, indent=2, sort_keys=True)
    )

print("Worker runtime versions match frozen Step 4A exactly.")

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–23."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 24 is already present in the completion registry."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 24 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 source root differs before Step 5A."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)

# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 24 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

condition_plan["ConditionOrder"] = parse_int(
    condition_plan["ConditionOrder"],
    "condition_plan.ConditionOrder",
)
condition_plan["NoisePercent"] = parse_int(
    condition_plan["NoisePercent"],
    "condition_plan.NoisePercent",
)
condition_plan["RepetitionSeed"] = parse_int(
    condition_plan["RepetitionSeed"],
    "condition_plan.RepetitionSeed",
)

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(condition_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain 270 conditions."
    )

if not np.array_equal(
    condition_plan["ConditionOrder"].to_numpy(dtype=np.int64),
    np.arange(1, EXPECTED_CONDITIONS + 1, dtype=np.int64),
):
    raise RuntimeError(
        "The frozen condition-order sequence is not canonical."
    )

if sorted(condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError(
        "The frozen noise-level set differs."
    )

if sorted(condition_plan["RepetitionSeed"].unique().tolist()) != REPETITION_SEEDS:
    raise RuntimeError(
        "The frozen repetition-seed set differs."
    )

if condition_plan["ConditionID"].duplicated(keep=False).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate condition IDs."
    )

if condition_plan.duplicated(
    subset=["NoisePercent", "RepetitionSeed"],
    keep=False,
).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate coordinates."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG-manifest row count differs."
    )

# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

# Project 24 Step 1B/2B chronology schema is BuildOrder / Build.
required_chronology_columns = {
    "BuildOrder",
    "Build",
}
if not required_chronology_columns.issubset(chronology.columns):
    raise RuntimeError(
        "Frozen Project 24 chronology is missing BuildOrder/Build."
    )

chronology["Build"] = parse_int(
    chronology["Build"],
    "chronology.Build",
)
chronology["BuildOrder"] = parse_int(
    chronology["BuildOrder"],
    "chronology.BuildOrder",
)

if not np.array_equal(
    chronology.sort_values(
        "BuildOrder",
        kind="mergesort",
    )["BuildOrder"].to_numpy(dtype=np.int64),
    np.arange(1, len(chronology) + 1, dtype=np.int64),
):
    raise RuntimeError(
        "Frozen Project 24 BuildOrder is not exactly 1..N."
    )

build_order_map = (
    chronology.set_index("Build")[
        "BuildOrder"
    ]
    .astype(int)
    .to_dict()
)

ordered_builds = (
    chronology.sort_values(
        "BuildOrder",
        kind="mergesort",
    )["Build"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

# Project 24 Step 2A/2B build-entity schema is Build / EntityId.
required_build_entity_columns = {
    "Build",
    "EntityId",
}
if not required_build_entity_columns.issubset(build_entity.columns):
    raise RuntimeError(
        "Frozen Project 24 build-entity map is missing Build/EntityId."
    )

build_entity["Build"] = parse_int(
    build_entity["Build"],
    "build_entity.Build",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "Build"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["Build"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

# --------------------------------------------------------------------------------------------------
# 7B. PRECOMPUTE THE ACCELERATED REC ENGINE
# --------------------------------------------------------------------------------------------------

print("Precomputing the vectorized verdict-dependent REC engine.")

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing the Step-2B-frozen InferredTestOrder column."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

combined_history = pd.DataFrame({
    "CombinedRowIndex": np.arange(
        EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS,
        dtype=np.int64,
    ),
    "Build": np.concatenate((
        raw_training["Build"].to_numpy(dtype=np.int64),
        raw_evaluation["Build"].to_numpy(dtype=np.int64),
    )),
    "Test": np.concatenate((
        raw_training["Test"].to_numpy(dtype=np.int64),
        raw_evaluation["Test"].to_numpy(dtype=np.int64),
    )),
    "InferredTestOrder": np.concatenate((
        raw_training["InferredTestOrder"].to_numpy(dtype=np.int64),
        raw_evaluation["InferredTestOrder"].to_numpy(dtype=np.int64),
    )),
})

if combined_history.duplicated(
    subset=["Test", "InferredTestOrder"],
    keep=False,
).any():
    raise RuntimeError(
        "Accelerated REC history contains duplicate frozen per-test order keys."
    )

# Project 24 must use the exact per-test execution order frozen by Step 2B.
# Project 24 has 23 frozen timestamp-tie groups; InferredTestOrder is the authoritative per-test order from Step 2B.
combined_history = (
    combined_history.sort_values(
        ["Test", "InferredTestOrder"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

accelerated_history_combined_indices = combined_history[
    "CombinedRowIndex"
].to_numpy(dtype=np.int64)
accelerated_history_builds = combined_history[
    "Build"
].to_numpy(dtype=np.int64)
accelerated_history_tests = combined_history[
    "Test"
].to_numpy(dtype=np.int64)

accelerated_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        accelerated_history_tests[1:]
        != accelerated_history_tests[:-1]
    ).astype(np.int64) + 1,
))
accelerated_group_ends = np.concatenate((
    accelerated_group_starts[1:],
    np.array([len(combined_history)], dtype=np.int64),
))
accelerated_group_count = len(accelerated_group_starts)

if accelerated_group_count != int(combined_history["Test"].nunique()):
    raise RuntimeError(
        "Accelerated REC test-group count differs."
    )

history_key_index = pd.MultiIndex.from_arrays([
    accelerated_history_builds,
    accelerated_history_tests,
])

if not history_key_index.is_unique:
    raise RuntimeError(
        "Accelerated REC history contains duplicate Build-Test keys."
    )

model_history_positions = history_key_index.get_indexer(
    model_key_index
)

if (model_history_positions < 0).any():
    raise RuntimeError(
        "Accelerated REC history does not cover every model row."
    )

model_group_indices = np.searchsorted(
    accelerated_group_starts,
    model_history_positions,
    side="right",
) - 1
model_local_positions = (
    model_history_positions
    - accelerated_group_starts[model_group_indices]
)

accelerated_requested_model_indices = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]
accelerated_requested_local_positions = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]

request_order = np.lexsort((
    model_local_positions,
    model_group_indices,
))
ordered_group_indices = model_group_indices[request_order]
request_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        ordered_group_indices[1:]
        != ordered_group_indices[:-1]
    ).astype(np.int64) + 1,
))
request_group_ends = np.concatenate((
    request_group_starts[1:],
    np.array([len(request_order)], dtype=np.int64),
))

for request_start, request_end in zip(
    request_group_starts,
    request_group_ends,
):
    selected = request_order[request_start:request_end]
    group_index = int(model_group_indices[selected[0]])
    accelerated_requested_model_indices[group_index] = selected.astype(
        np.int64,
        copy=False,
    )
    accelerated_requested_local_positions[group_index] = model_local_positions[
        selected
    ].astype(np.int64, copy=False)

accelerated_entity_values = np.sort(
    build_entity["EntityId"].unique().astype(np.int64)
)
accelerated_entity_count = len(accelerated_entity_values)
accelerated_entity_to_dense = {
    int(entity_id): dense_index
    for dense_index, entity_id in enumerate(accelerated_entity_values)
}

accelerated_build_values = np.asarray(
    ordered_builds,
    dtype=np.int64,
)
accelerated_build_to_dense = {
    int(build_id): dense_index
    for dense_index, build_id in enumerate(accelerated_build_values)
}
accelerated_build_entity_arrays = [
    np.empty(0, dtype=np.int32)
    for _ in accelerated_build_values
]

for build_id, entity_ids in build_entity.groupby(
    "Build",
    sort=False,
)["EntityId"]:
    build_dense = accelerated_build_to_dense[int(build_id)]
    accelerated_build_entity_arrays[build_dense] = np.asarray(
        sorted({
            accelerated_entity_to_dense[int(entity_id)]
            for entity_id in entity_ids
        }),
        dtype=np.int32,
    )

accelerated_history_build_dense_indices = np.asarray([
    accelerated_build_to_dense[int(build_id)]
    for build_id in accelerated_history_builds
], dtype=np.int32)

accelerated_dependent_feature_indices = np.asarray([
    REC_FEATURES.index(feature)
    for feature in VERDICT_DEPENDENT_REC
], dtype=np.int64)
accelerated_anchor_dependent_all = anchor_values_all[
    :, accelerated_dependent_feature_indices
]

# Exact clean-equivalence self-test before any full condition is allowed.
accelerated_clean_combined_verdict = np.concatenate((
    clean_raw_training_verdict,
    clean_raw_evaluation_verdict,
)).astype(np.int16, copy=False)
accelerated_clean_reconstructed = reconstruct_dependent_rec_fast(
    accelerated_clean_combined_verdict
)
accelerated_clean_anchored = (
    accelerated_clean_reconstructed
    + accelerated_anchor_dependent_all
)
accelerated_clean_original = clean_original_rec_all[
    :, accelerated_dependent_feature_indices
]
accelerated_clean_mismatch_values = int((
    ~np.isclose(
        accelerated_clean_anchored,
        accelerated_clean_original,
        rtol=0,
        atol=1e-12,
    )
).sum())

if accelerated_clean_mismatch_values != 0:
    raise RuntimeError(
        "Accelerated REC engine failed the exact clean-data equivalence test."
    )

print(
    "Accelerated REC engine clean-equivalence mismatches:",
    accelerated_clean_mismatch_values,
)
print(
    "Accelerated REC groups / model rows / entities:",
    accelerated_group_count,
    "/",
    EXPECTED_MODEL_ROWS,
    "/",
    accelerated_entity_count,
)

# --------------------------------------------------------------------------------------------------
# 7B. WORKER SHARD FREEZE AND MASTER-WRITE GUARD
# --------------------------------------------------------------------------------------------------

worker_condition_plan = (
    condition_plan.loc[
        condition_plan["RepetitionSeed"].isin(WORKER_REPETITION_SEEDS)
    ]
    .sort_values("ConditionOrder", kind="mergesort")
    .reset_index(drop=True)
)

if len(worker_condition_plan) != EXPECTED_WORKER_CONDITIONS:
    raise RuntimeError(
        "Worker condition shard does not contain the expected number of conditions.\n"
        f"Expected: {EXPECTED_WORKER_CONDITIONS}\n"
        f"Actual:   {len(worker_condition_plan)}"
    )

if sorted(worker_condition_plan["RepetitionSeed"].unique().tolist()) != WORKER_REPETITION_SEEDS:
    raise RuntimeError("Worker repetition-seed shard differs from its frozen assignment.")

if sorted(worker_condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError("Worker noise-level set differs from the frozen 9-level protocol.")

MASTER_WRITE_GUARD_PATHS = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    STEP5A_VALIDATION_PATH,
    STEP5A_REPORT_PATH,
    MASTER_RUN_PROGRESS_PATH,
    STEP5A_STATUS_PATH,
    STEP5A_CHECKPOINT_PATH,
]
master_write_guard_before = {
    str(path): path_guard_state(path)
    for path in MASTER_WRITE_GUARD_PATHS
}

print("Worker shard frozen:", WORKER_REPETITION_SEEDS)
print("Worker conditions:", len(worker_condition_plan))
print("Master Step 5A write guard armed for", len(MASTER_WRITE_GUARD_PATHS), "paths.")

# --------------------------------------------------------------------------------------------------
# 8. CHECKPOINT SCAN AND ASSIGNED CONDITION RUNNER
# --------------------------------------------------------------------------------------------------

FULL_RAW_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
FULL_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
INCOMPLETE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

raw_training_hash_before = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_before = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_before = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_before = sha256_file(MODEL_EVALUATION_COHORT_PATH)

expected_condition_files = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}

def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != expected_condition_files:
        return None

    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None

    if completion.get("Status") != CONDITION_STATUS:
        return None
    if summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key:
        return None
    if summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None

    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None

    for item in output_manifest:
        path = Path(item.get("Path", ""))
        if path.parent != condition_dir:
            return None
        if not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None

    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None

    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None

    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None

    condition_manifest = directory_manifest(condition_dir)
    if len(condition_manifest) != EXPECTED_FILES_PER_CONDITION:
        return None

    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(condition_manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "ModelFitRows": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "ConditionSeconds": float(summary["ConditionSeconds"]),
        "RandomScoreSHA256": str(fingerprints["Random"]["ScoreSHA256"]),
        "RandomRankSHA256": str(fingerprints["Random"]["RankSHA256"]),
        "QTFAvgScoreSHA256": str(fingerprints["QTF-Avg"]["ScoreSHA256"]),
        "QTFAvgRankSHA256": str(fingerprints["QTF-Avg"]["RankSHA256"]),
    }

def quarantine_incomplete_condition(condition_dir):
    condition_dir = Path(condition_dir)

    if not condition_dir.exists():
        return None

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = INCOMPLETE_BACKUP_ROOT / f"{condition_dir.name}__{timestamp}"
    shutil.move(str(condition_dir), str(destination))
    return destination

print("\nScanning existing condition checkpoints...")

valid_existing = {}
invalid_existing = []

for plan_row in worker_condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is not None:
        valid_existing[condition_key] = validated
    elif condition_dir.exists():
        invalid_existing.append(condition_key)

print("Valid completed conditions:", len(valid_existing))
print("Incomplete/invalid condition directories:", len(invalid_existing))
print("Pending assigned conditions:", EXPECTED_WORKER_CONDITIONS - len(valid_existing))

for condition_key in invalid_existing:
    backup = quarantine_incomplete_condition(
        FULL_RAW_RESULT_ROOT / condition_key
    )
    print("Preserved incomplete condition in:", backup)

# The seed-1 worker owns the shared accelerated-equivalence audit.
# Other workers may start simultaneously but wait here until that exact two-row PASS audit exists.
equivalence_records_by_key = {}

if WORKER_IS_SMOKE_OWNER:
    for equivalence_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_row = condition_plan.loc[
            condition_plan["ConditionID"].eq(equivalence_key)
        ]
        if len(equivalence_row) != 1:
            raise RuntimeError(
                f"Could not uniquely resolve smoke-equivalence condition {equivalence_key}."
            )
        equivalence_plan_row = next(equivalence_row.itertuples(index=False))
        equivalence_dir = FULL_RAW_RESULT_ROOT / equivalence_key
        if validate_completed_condition(equivalence_dir, equivalence_plan_row) is not None:
            equivalence_record = compare_full_condition_to_smoke(
                equivalence_key,
                equivalence_dir,
            )
            if not equivalence_record["Pass"]:
                raise RuntimeError(
                    f"Existing accelerated condition {equivalence_key} differs from Step 4B."
                )
            equivalence_records_by_key[equivalence_key] = equivalence_record

    smoke_first_plan = worker_condition_plan.loc[
        worker_condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
    ].copy()
    smoke_first_plan["__SmokeOrder"] = smoke_first_plan["ConditionID"].map({
        key: index
        for index, key in enumerate(SMOKE_EQUIVALENCE_KEYS)
    })
    smoke_first_plan = smoke_first_plan.sort_values(
        "__SmokeOrder",
        kind="mergesort",
    ).drop(columns="__SmokeOrder")
    remaining_plan = worker_condition_plan.loc[
        ~worker_condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
    ].sort_values("ConditionOrder", kind="mergesort")
    execution_plan = pd.concat(
        [smoke_first_plan, remaining_plan],
        ignore_index=True,
    )
else:
    shared_equivalence = wait_for_shared_equivalence()
    equivalence_records_by_key = {
        str(row.ConditionKey): {"Pass": True}
        for row in shared_equivalence.itertuples(index=False)
    }
    execution_plan = worker_condition_plan.copy()

if len(execution_plan) != EXPECTED_WORKER_CONDITIONS:
    raise RuntimeError(
        "Worker accelerated execution plan does not contain its exact assigned conditions."
    )

if WORKER_IS_SMOKE_OWNER and len(equivalence_records_by_key) == len(SMOKE_EQUIVALENCE_KEYS):
    atomic_csv(
        ACCELERATED_EQUIVALENCE_PATH,
        pd.DataFrame(equivalence_records_by_key.values()).sort_values(
            "ConditionKey",
            kind="mergesort",
        ),
    )

rng_parquet_file = pq.ParquetFile(
    RNG_MANIFEST_PATH
)

if int(rng_parquet_file.metadata.num_rows) != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs before worker execution."
    )

if int(rng_parquet_file.metadata.num_row_groups) != 30:
    raise RuntimeError(
        "Frozen RNG manifest must contain exactly 30 one-seed row groups."
    )

full_execution_started = time.perf_counter()
completed_this_run = 0
skipped_valid = 0
current_rng_seed = None
flip_uniform = None
sampled_failure_subtype = None
random_scores = None

for worker_position, plan_row in enumerate(execution_plan.itertuples(index=False), start=1):
    condition_order = int(plan_row.ConditionOrder)
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key

    already_valid = validate_completed_condition(condition_dir, plan_row)
    if already_valid is not None:
        skipped_valid += 1
        print(
            f"[worker {worker_position}/{EXPECTED_WORKER_CONDITIONS} | global {condition_order}/{EXPECTED_CONDITIONS}] "
            f"Skipping validated checkpoint {condition_key}"
        )
        continue

    if (
        condition_key not in SMOKE_EQUIVALENCE_KEYS
        and set(equivalence_records_by_key) != set(SMOKE_EQUIVALENCE_KEYS)
    ):
        raise RuntimeError(
            "The accelerated engine must pass both frozen smoke-output equivalence checks "
            "before any other full condition is executed."
        )

    if repetition_seed != current_rng_seed:
        print(f"\nLoading deterministic RNG stream for seed {repetition_seed}.")

        rng_seed_frame = rng_parquet_file.read_row_group(
            repetition_seed - 1,
            columns=[
                "RepetitionSeed",
                "RawTrainingRowOrder",
                "FlipUniform",
                "SampledFailureSubtype",
            ],
        ).to_pandas()

        if (
            rng_seed_frame["RepetitionSeed"].nunique() != 1
            or int(rng_seed_frame["RepetitionSeed"].iloc[0]) != repetition_seed
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row-group seed identity differs."
            )
        rng_seed_frame[raw_order_column] = parse_int(
            rng_seed_frame[raw_order_column],
            f"rng_seed_{repetition_seed}.RawTrainingRowOrder",
        )
        rng_seed_frame = (
            rng_seed_frame.sort_values(
                raw_order_column,
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        if len(rng_seed_frame) != EXPECTED_RAW_TRAIN_ROWS:
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG stream row count differs."
            )

        if not np.array_equal(
            rng_seed_frame[raw_order_column].to_numpy(dtype=np.int64),
            np.arange(1, EXPECTED_RAW_TRAIN_ROWS + 1, dtype=np.int64),
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row order differs."
            )

        flip_uniform = rng_seed_frame["FlipUniform"].to_numpy(dtype=np.float64)
        sampled_failure_subtype = rng_seed_frame[
            "SampledFailureSubtype"
        ].to_numpy(dtype=np.int16)

        if not np.isfinite(flip_uniform).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: non-finite flip uniforms."
            )
        if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
            raise RuntimeError(
                f"Seed {repetition_seed}: flip uniforms outside [0,1)."
            )

        # Exact Step 4B Random-baseline assignment:
        # draw per build after sorting that build's rows by Test, then place scores
        # back into the fixed evaluation-row positions.
        random_scores = np.empty(EXPECTED_MODEL_EVAL_ROWS, dtype=np.float64)
        eval_build_array = evaluation_meta_base["Build"].to_numpy(dtype=np.int64)
        eval_test_array = evaluation_meta_base["Test"].to_numpy(dtype=np.int64)

        for build_id in sorted(evaluation_meta_base["Build"].unique()):
            build_indices = np.flatnonzero(
                eval_build_array == int(build_id)
            )
            ordered_indices = build_indices[
                np.argsort(
                    eval_test_array[build_indices],
                    kind="mergesort",
                )
            ]
            random_scores[ordered_indices] = np.random.default_rng(
                deterministic_random_build_seed(
                    repetition_seed,
                    int(build_id),
                )
            ).random(len(ordered_indices))

        if not np.isfinite(random_scores).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: Random baseline scores are non-finite."
            )

        current_rng_seed = repetition_seed
        del rng_seed_frame
        gc.collect()

    condition_started = time.perf_counter()

    print("\n" + "-" * 110)
    print(
        f"[worker {worker_position}/{EXPECTED_WORKER_CONDITIONS} | global {condition_order}/{EXPECTED_CONDITIONS}] Running {condition_key}"
    )
    print("-" * 110)

    condition_dir.mkdir(parents=True, exist_ok=True)

    ranking_path = condition_dir / "rankings.csv.gz"
    build_metrics_path = condition_dir / "build_metrics.csv"
    project_runs_path = condition_dir / "project_runs.csv"
    model_fits_path = condition_dir / "model_fits.csv"
    training_medians_path = condition_dir / "training_medians.csv"
    condition_audit_path = condition_dir / "condition_audit.csv"
    condition_summary_path = condition_dir / "condition_summary.json"
    completion_marker_path = condition_dir / "COMPLETE.json"

    flip_mask = flip_uniform < (noise_percent / 100.0)
    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = flip_mask & (clean_raw_training_verdict == 0)
    failure_to_pass_mask = flip_mask & (clean_raw_training_verdict != 0)

    noisy_raw_training_verdict[pass_to_failure_mask] = (
        sampled_failure_subtype[pass_to_failure_mask]
    )
    noisy_raw_training_verdict[failure_to_pass_mask] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(plan_row.FlipMaskSHA256)
    expected_noisy_raw_sha256 = str(plan_row.NoisyRawVerdictSHA256)
    expected_noisy_model_sha256 = str(plan_row.NoisyModelVerdictSHA256)

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )
    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )
    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (noisy_model_training_verdict != clean_model_training_verdict).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(noisy_model_training_binary.sum())

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    rec_started = time.perf_counter()

    if noise_percent == 0:
        # The 0% REC matrix is already frozen and exact. Reusing it avoids
        # thirty identical full-history reconstructions.
        condition_numeric_all = all_base_numeric.copy()
        dependent_rec_changes = 0
        independent_rec_changes = 0
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS
    else:
        condition_combined_verdict = np.concatenate((
            noisy_raw_training_verdict,
            clean_raw_evaluation_verdict,
        )).astype(np.int16, copy=False)

        reconstructed_dependent = reconstruct_dependent_rec_fast(
            condition_combined_verdict
        )
        anchored_dependent = (
            reconstructed_dependent
            + accelerated_anchor_dependent_all
        )

        condition_numeric_all = all_base_numeric.copy()
        condition_numeric_all[:, dependent_predictor_indices] = (
            anchored_dependent
        )

        dependent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, dependent_predictor_indices],
                all_base_numeric[:, dependent_predictor_indices],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        independent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, independent_predictor_indices],
                all_base_numeric[:, independent_predictor_indices],
                rtol=0,
                atol=0,
                equal_nan=True,
            )
        ).sum())
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS

        if independent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: preserved independent REC predictors changed."
            )

    rec_seconds = time.perf_counter() - rec_started

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    if noise_percent == 0:
        zero_rec_mismatches = int((
            ~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        if zero_rec_mismatches != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition did not reproduce clean REC."
            )
        if number_flipped != 0 or model_label_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed labels."
            )
        if dependent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed dependent REC."
            )
    else:
        if number_flipped <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no dependent REC."
            )

    medians = np.nanmedian(condition_training_numeric, axis=0)
    nonfinite_median_indices = np.flatnonzero(~np.isfinite(medians))
    if len(nonfinite_median_indices) != 0:
        bad_features = [predictor_columns[index] for index in nonfinite_median_indices]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(condition_training_numeric)
    evaluation_missing_mask = ~np.isfinite(condition_evaluation_numeric)

    if training_missing_mask.any():
        row_indices, column_indices = np.where(training_missing_mask)
        condition_training_numeric[row_indices, column_indices] = medians[
            column_indices
        ]
    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(evaluation_missing_mask)
        condition_evaluation_numeric[row_indices, column_indices] = medians[
            column_indices
        ]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite."
        )
    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite."
        )

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )
            fit_seconds = time.perf_counter() - fit_started
            technique_scores[technique] = positive_probability(
                model,
                condition_evaluation_numeric,
            )
            fit_status = "PASS_MODEL_FIT"
            fit_error = ""
        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)
            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })
            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps([
                int(value)
                for value in np.asarray(model.classes_).tolist()
            ]),
            "Status": fit_status,
            "Error": fit_error,
        })
        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)
    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )
    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :, predictor_index["REC_LastFailureAge"]
    ].astype(float)
    condition_eval_qtf = condition_evaluation_numeric[
        :, predictor_index["REC_TotalAvgExeTime"]
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = -condition_eval_last_failure_age
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []
    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(ranking_frames, ignore_index=True)
    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )
    if sorted(rankings["Technique"].unique().tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )
    if not rankings.groupby("Technique").size().eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )
    if rankings.duplicated(
        subset=["Technique", "Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(rankings)
    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )
    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )
    if sorted(project_runs["Technique"].tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    build_metric_values = build_metrics[["APFDc", "APFD"]].to_numpy(dtype=float)
    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )
    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_values = project_runs[[
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]].to_numpy(dtype=float)
    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )
    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(reconstructed_row_count),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    baseline_fingerprints = {}
    for technique in ["Random", "QTF-Avg"]:
        baseline_rows = (
            rankings.loc[
                rankings["Technique"].eq(technique),
                ["Build", "Test", "Score", "Rank"],
            ]
            .sort_values(["Build", "Test"], kind="mergesort")
            .reset_index(drop=True)
        )
        baseline_fingerprints[technique] = {
            "Rows": int(len(baseline_rows)),
            "KeySHA256": hashlib.sha256(
                np.ascontiguousarray(
                    baseline_rows[["Build", "Test"]].to_numpy(dtype=np.int64)
                ).tobytes(order="C")
            ).hexdigest(),
            "ScoreSHA256": sha256_array(
                baseline_rows["Score"].to_numpy(dtype=np.float64),
                "<f8",
            ),
            "RankSHA256": sha256_array(
                baseline_rows["Rank"].to_numpy(dtype=np.int64),
                "<i8",
            ),
        }

    atomic_csv(ranking_path, rankings, compression="gzip")
    atomic_csv(build_metrics_path, build_metrics)
    atomic_csv(project_runs_path, project_runs)
    atomic_csv(model_fits_path, model_fits)
    atomic_csv(training_medians_path, training_medians)
    atomic_csv(condition_audit_path, condition_audit)

    if condition_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_record = compare_full_condition_to_smoke(
            condition_key,
            condition_dir,
        )
        if not equivalence_record["Pass"]:
            print("\nAccelerated-engine equivalence failure:")
            display(pd.DataFrame([equivalence_record]))
            raise RuntimeError(
                f"{condition_key}: accelerated outputs differ from the frozen Step 4B outputs."
            )
        equivalence_records_by_key[condition_key] = equivalence_record
        if WORKER_IS_SMOKE_OWNER:
            atomic_csv(
                ACCELERATED_EQUIVALENCE_PATH,
                pd.DataFrame(equivalence_records_by_key.values()).sort_values(
                    "ConditionKey",
                    kind="mergesort",
                ),
            )
        print(
            "  Frozen smoke-output equivalence: PASS |",
            condition_key,
        )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]
    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "BaselineFingerprints": baseline_fingerprints,
        "OutputManifest": condition_output_manifest,
    }
    atomic_json(condition_summary_path, condition_summary)

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }
    atomic_json(completion_marker_path, completion_marker)

    validated_after_write = validate_completed_condition(
        condition_dir,
        plan_row,
    )
    if validated_after_write is None:
        raise RuntimeError(
            f"{condition_key}: completed condition did not pass readback validation."
        )

    completed_this_run += 1
    completed_total = skipped_valid + completed_this_run

    atomic_json(
        RUN_PROGRESS_PATH,
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": f"PROJECT_24_STEP5A_WORKER_{WORKER_TAG}_IN_PROGRESS",
            "UpdatedAtUTC": datetime.now(timezone.utc).isoformat(),
            "CompletedConditions": completed_total,
            "ExpectedConditions": EXPECTED_WORKER_CONDITIONS,
            "LastCompletedCondition": condition_key,
            "LastCompletedConditionOrder": condition_order,
            "ResumeSafe": True,
            "RegistryModified": False,
            "PriorProjectConditionOutputsAccessed": False,
        },
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del condition_numeric_all
    if noise_percent > 0:
        del condition_combined_verdict
        del reconstructed_dependent
        del anchored_dependent
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()

# --------------------------------------------------------------------------------------------------
# 9. FINAL ASSIGNED-SHARD REVALIDATION
# --------------------------------------------------------------------------------------------------

print(f"\nValidating all {EXPECTED_WORKER_CONDITIONS} assigned conditions for {WORKER_TAG}.")

inventory_records = []
condition_audits = []
baseline_records = []
worker_raw_manifest_records = []
model_fit_rows = 0
build_metric_rows = 0
project_run_rows = 0

for plan_row in worker_condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is None:
        raise RuntimeError(
            f"Worker final validation failed for {condition_key}."
        )

    inventory_records.append(validated)
    condition_audits.append(pd.read_csv(condition_dir / "condition_audit.csv"))
    model_fit_rows += len(pd.read_csv(condition_dir / "model_fits.csv"))
    build_metric_rows += len(pd.read_csv(condition_dir / "build_metrics.csv"))
    project_run_rows += len(pd.read_csv(condition_dir / "project_runs.csv"))

    summary = load_json(condition_dir / "condition_summary.json")
    for technique in ["Random", "QTF-Avg"]:
        fingerprint = summary["BaselineFingerprints"][technique]
        baseline_records.append({
            "ConditionKey": condition_key,
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "Rows": int(fingerprint["Rows"]),
            "KeySHA256": str(fingerprint["KeySHA256"]),
            "ScoreSHA256": str(fingerprint["ScoreSHA256"]),
            "RankSHA256": str(fingerprint["RankSHA256"]),
        })

    condition_manifest = directory_manifest(condition_dir)
    for item in condition_manifest.itertuples(index=False):
        worker_raw_manifest_records.append({
            "RelativePath": f"{condition_key}/{item.RelativePath}",
            "Bytes": int(item.Bytes),
            "SHA256": str(item.SHA256),
        })

worker_inventory = pd.DataFrame(inventory_records).sort_values(
    "ConditionOrder",
    kind="mergesort",
).reset_index(drop=True)
combined_condition_audit = pd.concat(condition_audits, ignore_index=True)
baseline_fingerprints = pd.DataFrame(baseline_records)
worker_raw_manifest = pd.DataFrame(
    worker_raw_manifest_records,
    columns=DIRECTORY_MANIFEST_COLUMNS,
).sort_values("RelativePath", kind="mergesort").reset_index(drop=True)
worker_raw_root_sha256 = directory_root_hash(worker_raw_manifest)
worker_raw_files = int(len(worker_raw_manifest))
worker_raw_bytes = int(worker_raw_manifest["Bytes"].sum())

baseline_invariance_records = []
for repetition_seed in WORKER_REPETITION_SEEDS:
    for technique in ["Random", "QTF-Avg"]:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints["RepetitionSeed"].eq(repetition_seed)
            & baseline_fingerprints["Technique"].eq(technique)
        ]
        key_variants = int(rows["KeySHA256"].nunique())
        score_variants = int(rows["ScoreSHA256"].nunique())
        rank_variants = int(rows["RankSHA256"].nunique())
        baseline_invariance_records.append({
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "Conditions": int(len(rows)),
            "KeyVariantsAcrossNoise": key_variants,
            "ScoreVariantsAcrossNoise": score_variants,
            "RankVariantsAcrossNoise": rank_variants,
            "Pass": bool(
                len(rows) == len(NOISE_LEVELS)
                and key_variants == 1
                and score_variants == 1
                and rank_variants == 1
            ),
        })

worker_baseline_invariance = pd.DataFrame(baseline_invariance_records)
baseline_invariance_failures = int((~worker_baseline_invariance["Pass"]).sum())
worker_qtf_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "ScoreSHA256",
    ].nunique()
)
worker_qtf_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "RankSHA256",
    ].nunique()
)

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
]
positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].gt(0)
]
noise_plan_hash_mismatches = int(
    combined_condition_audit["ExpectedFlipMaskSHA256"].ne(
        combined_condition_audit["ActualFlipMaskSHA256"]
    ).sum()
    + combined_condition_audit["ExpectedNoisyRawVerdictSHA256"].ne(
        combined_condition_audit["ActualNoisyRawVerdictSHA256"]
    ).sum()
    + combined_condition_audit["ExpectedNoisyModelVerdictSHA256"].ne(
        combined_condition_audit["ActualNoisyModelVerdictSHA256"]
    ).sum()
)

shared_equivalence = load_valid_shared_equivalence()
shared_equivalence_rows = 0 if shared_equivalence is None else len(shared_equivalence)

registry_sha256_after = sha256_file(REGISTRY_PATH)
current_source_rows_after = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })
source_root_sha256_after = source_root_hash(pd.DataFrame(current_source_rows_after))

master_write_guard_after = {
    str(path): path_guard_state(path)
    for path in MASTER_WRITE_GUARD_PATHS
}
master_write_guard_failures = sum(
    int(master_write_guard_before[path] != master_write_guard_after[path])
    for path in master_write_guard_before
)

validation_records = []
add_check(validation_records, "Worker assigned seeds", WORKER_REPETITION_SEEDS, sorted(worker_inventory["RepetitionSeed"].unique().tolist()), sorted(worker_inventory["RepetitionSeed"].unique().tolist()) == WORKER_REPETITION_SEEDS)
add_check(validation_records, "Worker conditions", EXPECTED_WORKER_CONDITIONS, len(worker_inventory), len(worker_inventory) == EXPECTED_WORKER_CONDITIONS)
add_check(validation_records, "Worker noise levels", NOISE_LEVELS, sorted(worker_inventory["NoisePercent"].unique().tolist()), sorted(worker_inventory["NoisePercent"].unique().tolist()) == NOISE_LEVELS)
add_check(validation_records, "Duplicate condition keys", 0, int(worker_inventory["ConditionKey"].duplicated(keep=False).sum()), not worker_inventory["ConditionKey"].duplicated(keep=False).any())
add_check(validation_records, "Files per condition", EXPECTED_FILES_PER_CONDITION, sorted(worker_inventory["Files"].unique().tolist()), worker_inventory["Files"].eq(EXPECTED_FILES_PER_CONDITION).all())
add_check(validation_records, "Worker raw files", EXPECTED_WORKER_RAW_FILES, worker_raw_files, worker_raw_files == EXPECTED_WORKER_RAW_FILES)
add_check(validation_records, "Worker ranking rows", EXPECTED_WORKER_RANKING_ROWS, int(worker_inventory["RankingRows"].sum()), int(worker_inventory["RankingRows"].sum()) == EXPECTED_WORKER_RANKING_ROWS)
add_check(validation_records, "Worker build-metric rows", EXPECTED_WORKER_BUILD_METRIC_ROWS, build_metric_rows, build_metric_rows == EXPECTED_WORKER_BUILD_METRIC_ROWS)
add_check(validation_records, "Worker project-run rows", EXPECTED_WORKER_PROJECT_RUN_ROWS, project_run_rows, project_run_rows == EXPECTED_WORKER_PROJECT_RUN_ROWS)
add_check(validation_records, "Worker model fits", EXPECTED_WORKER_MODEL_FITS, model_fit_rows, model_fit_rows == EXPECTED_WORKER_MODEL_FITS)
add_check(validation_records, "Worker condition-audit rows", EXPECTED_WORKER_CONDITIONS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_WORKER_CONDITIONS)
add_check(validation_records, "Worker training-median rows", EXPECTED_WORKER_TRAINING_MEDIAN_ROWS, int(worker_inventory["TrainingMedianRows"].sum()), int(worker_inventory["TrainingMedianRows"].sum()) == EXPECTED_WORKER_TRAINING_MEDIAN_ROWS)
add_check(validation_records, "Zero-noise conditions", len(WORKER_REPETITION_SEEDS), len(zero_audit), len(zero_audit) == len(WORKER_REPETITION_SEEDS))
add_check(validation_records, "Zero-noise raw flips", 0, int(zero_audit["NumberFlipped"].sum()), int(zero_audit["NumberFlipped"].sum()) == 0)
add_check(validation_records, "Zero-noise model-label changes", 0, int(zero_audit["ModelLabelChanges"].sum()), int(zero_audit["ModelLabelChanges"].sum()) == 0)
add_check(validation_records, "Zero-noise dependent REC changes", 0, int(zero_audit["DependentRECChanges"].sum()), int(zero_audit["DependentRECChanges"].sum()) == 0)
add_check(validation_records, "Positive-noise raw-change violations", 0, int(positive_audit["NumberFlipped"].le(0).sum()), not positive_audit["NumberFlipped"].le(0).any())
add_check(validation_records, "Positive-noise model-change violations", 0, int(positive_audit["ModelLabelChanges"].le(0).sum()), not positive_audit["ModelLabelChanges"].le(0).any())
add_check(validation_records, "Positive-noise dependent-REC violations", 0, int(positive_audit["DependentRECChanges"].le(0).sum()), not positive_audit["DependentRECChanges"].le(0).any())
add_check(validation_records, "Independent REC changes", 0, int(combined_condition_audit["IndependentRECChanges"].sum()), int(combined_condition_audit["IndependentRECChanges"].sum()) == 0)
add_check(validation_records, "Independent reconstruction mismatches", 0, int(combined_condition_audit["IndependentReconstructionMismatches"].sum()), int(combined_condition_audit["IndependentReconstructionMismatches"].sum()) == 0)
add_check(validation_records, "Noise-plan hash mismatches", 0, noise_plan_hash_mismatches, noise_plan_hash_mismatches == 0)
add_check(validation_records, "Baseline invariance failures", 0, baseline_invariance_failures, baseline_invariance_failures == 0)
add_check(validation_records, "Worker QTF score variants", 1, worker_qtf_score_variants, worker_qtf_score_variants == 1)
add_check(validation_records, "Worker QTF rank variants", 1, worker_qtf_rank_variants, worker_qtf_rank_variants == 1)
add_check(validation_records, "Shared accelerated smoke-equivalence rows", 2, shared_equivalence_rows, shared_equivalence_rows == 2)
add_check(validation_records, "Accelerated clean REC mismatches", 0, accelerated_clean_mismatch_values, accelerated_clean_mismatch_values == 0)
add_check(validation_records, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_root_sha256_after, source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation_records, "Completion registry unchanged", registry_sha256_before, registry_sha256_after, registry_sha256_after == registry_sha256_before)
add_check(validation_records, "Master Step 5A write-guard failures", 0, master_write_guard_failures, master_write_guard_failures == 0)
add_check(validation_records, "Registry Project 24 rows", 0, int(registry_project_numbers.eq(PROJECT_NUMBER).sum()), int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) == 0)

worker_validation = pd.DataFrame(validation_records)
failed_validation = worker_validation.loc[~worker_validation["Pass"]]

print("\nWorker validation:")
display(worker_validation)

if not failed_validation.empty:
    print("\nFailed worker checks:")
    display(failed_validation)
    raise RuntimeError(
        f"PROJECT 24 STEP 5A WORKER {WORKER_TAG} VALIDATION FAILED."
    )

# --------------------------------------------------------------------------------------------------
# 10. FREEZE PRIVATE WORKER CHECKPOINT ONLY
# --------------------------------------------------------------------------------------------------

atomic_csv(WORKER_INVENTORY_PATH, worker_inventory)
atomic_csv(WORKER_RAW_MANIFEST_PATH, worker_raw_manifest)
atomic_csv(WORKER_BASELINE_INVARIANCE_PATH, worker_baseline_invariance)
atomic_csv(WORKER_COMBINED_AUDIT_PATH, combined_condition_audit)
atomic_csv(WORKER_VALIDATION_PATH, worker_validation)

worker_execution_seconds = time.perf_counter() - full_execution_started
completed_at_utc = datetime.now(timezone.utc).isoformat()

worker_output_paths = [
    WORKER_INVENTORY_PATH,
    WORKER_RAW_MANIFEST_PATH,
    WORKER_BASELINE_INVARIANCE_PATH,
    WORKER_COMBINED_AUDIT_PATH,
    WORKER_VALIDATION_PATH,
]
worker_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in worker_output_paths
]

worker_report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "WorkerTag": WORKER_TAG,
    "WorkerNotebookName": WORKER_NOTEBOOK_NAME,
    "WorkerCodeRevision": WORKER_CODE_REVISION,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "RECImplementation": EXPECTED_REC_IMPLEMENTATION,
    "Step3ACodeRevision": EXPECTED_STEP3A_CODE_REVISION,
    "RuntimeVersions": actual_runtime_versions,
    "Status": WORKER_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "AssignedSeeds": WORKER_REPETITION_SEEDS,
    "AssignedConditions": EXPECTED_WORKER_CONDITIONS,
    "CompletedConditions": len(worker_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "MLFits": model_fit_rows,
    "RankingRows": int(worker_inventory["RankingRows"].sum()),
    "BuildMetricRows": build_metric_rows,
    "ProjectRunRows": project_run_rows,
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(worker_inventory["TrainingMedianRows"].sum()),
    "WorkerRawFiles": worker_raw_files,
    "WorkerRawBytes": worker_raw_bytes,
    "WorkerRawRootSHA256": worker_raw_root_sha256,
    "AcceleratedCleanRECMismatches": accelerated_clean_mismatch_values,
    "SharedSmokeEquivalencePath": str(ACCELERATED_EQUIVALENCE_PATH),
    "SharedSmokeEquivalenceSHA256": sha256_file(ACCELERATED_EQUIVALENCE_PATH),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "ValidationChecks": len(worker_validation),
    "FailedValidationChecks": len(failed_validation),
    "CompletedThisInvocation": completed_this_run,
    "SkippedValidatedConditions": skipped_valid,
    "WorkerExecutionSecondsThisInvocation": float(worker_execution_seconds),
    "WorkerOutputManifest": worker_output_manifest,
    "SourceRootSHA256": source_root_sha256_after,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "MasterStep5AWriteGuardFailures": master_write_guard_failures,
    "Projects1To23Modified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "ResumeSafe": True,
    "MasterFinalizationRequired": True,
}
atomic_json(WORKER_REPORT_PATH, worker_report)

worker_checkpoint = {
    **worker_report,
    "CheckpointVersion": 1,
    "WorkerReportPath": str(WORKER_REPORT_PATH),
    "WorkerReportSHA256": sha256_file(WORKER_REPORT_PATH),
    "WorkerShardComplete": True,
    "OfficialStep5AComplete": False,
    "NextRequiredStep": "WAIT_FOR_ALL_SIX_WORKERS_THEN_MASTER_STEP5A_FINALIZATION",
}
atomic_json(WORKER_CHECKPOINT_PATH, worker_checkpoint)

worker_status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "WorkerTag": WORKER_TAG,
    "WorkerCodeRevision": WORKER_CODE_REVISION,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "Status": WORKER_STATUS,
    "AssignedSeeds": WORKER_REPETITION_SEEDS,
    "AssignedConditions": EXPECTED_WORKER_CONDITIONS,
    "CompletedConditions": len(worker_inventory),
    "WorkerRawFiles": worker_raw_files,
    "WorkerRawRootSHA256": worker_raw_root_sha256,
    "Checkpoint": str(WORKER_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(WORKER_CHECKPOINT_PATH),
    "OfficialStep5AComplete": False,
}
atomic_json(WORKER_STATUS_PATH, worker_status_payload)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "WorkerTag": WORKER_TAG,
        "Status": WORKER_STATUS,
        "UpdatedAtUTC": completed_at_utc,
        "CompletedConditions": EXPECTED_WORKER_CONDITIONS,
        "ExpectedConditions": EXPECTED_WORKER_CONDITIONS,
        "AssignedSeeds": WORKER_REPETITION_SEEDS,
        "CheckpointSHA256": sha256_file(WORKER_CHECKPOINT_PATH),
        "ResumeSafe": True,
        "OfficialStep5AComplete": False,
    },
)

checkpoint_readback = load_json(WORKER_CHECKPOINT_PATH)
if checkpoint_readback.get("Status") != WORKER_STATUS:
    raise RuntimeError("Worker checkpoint readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError("Completion registry changed during worker finalisation.")

final_master_guard_after = {
    str(path): path_guard_state(path)
    for path in MASTER_WRITE_GUARD_PATHS
}
if any(
    master_write_guard_before[path] != final_master_guard_after[path]
    for path in master_write_guard_before
):
    raise RuntimeError("A master Step 5A output changed during worker finalisation.")

print("\n" + "=" * 136)
print(f"=== PROJECT 24 STEP 5A WORKER {WORKER_TAG} RESULT ===")
print("=" * 136)
print("Project:", PROJECT_NAME)
print("Worker notebook:", WORKER_NOTEBOOK_NAME)
print("Worker code revision:", WORKER_CODE_REVISION)
print("Accelerated engine:", ACCELERATED_ENGINE_VERSION)
print("Assigned seeds:", WORKER_REPETITION_SEEDS)
print("Conditions:", len(worker_inventory), "/", EXPECTED_WORKER_CONDITIONS)
print("ML fits:", model_fit_rows, "/", EXPECTED_WORKER_MODEL_FITS)
print("Ranking rows:", int(worker_inventory["RankingRows"].sum()))
print("Build-metric rows:", build_metric_rows)
print("Project-run rows:", project_run_rows)
print("Condition-audit rows:", len(combined_condition_audit))
print("Training-median rows:", int(worker_inventory["TrainingMedianRows"].sum()))
print("Worker raw files:", worker_raw_files)
print("Worker raw bytes:", worker_raw_bytes)
print("Worker raw root SHA-256:", worker_raw_root_sha256)
print("Baseline invariance failures:", baseline_invariance_failures)
print("Master Step 5A write-guard failures:", master_write_guard_failures)
print("Validation checks:", len(worker_validation))
print("Failed checks:", len(failed_validation))
print("Completed this invocation:", completed_this_run)
print("Skipped validated conditions:", skipped_valid)
print("Runtime seconds this invocation:", round(worker_execution_seconds, 2))
print("Worker checkpoint:", WORKER_CHECKPOINT_PATH)
print("Worker checkpoint SHA-256:", sha256_file(WORKER_CHECKPOINT_PATH))
print("Official master Step 5A complete:", False)
print("STATUS:", WORKER_STATUS)
print("=" * 136)


=== PROJECT 24 STEP 5A PARALLEL WORKER: SEEDS 01-05 ===
Worker notebook: Thesis_p24_seed1-5
Assigned repetition seeds: [1, 2, 3, 4, 5]
Assigned conditions: 45
Mounting the shared thesis Google Drive.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restoring only SonarSource@sonarqube from the frozen TCP-CI archive into this worker runtime.
Project source files restored: 6
Worker runtime versions match frozen Step 4A exactly.

Loading frozen Project 24 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.
Precomputing the vectorized verdict-dependent REC engine.
Accelerated REC engine clean-equivalence mismatches: 0
Accelerated REC groups / model rows / entities: 2641 / 224550 / 14725
Worker shard frozen: [1, 2, 3, 4, 5]
Worker conditions: 45
Master Step 5A write guard armed for 12 paths.

Scanning existing condition checkpoints...
Valid completed conditions: 0
Incomplete/inva

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_00__seed_01
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 1777 | condition seconds: 113.68

--------------------------------------------------------------------------------------------------------------
[worker 2/45 | global 9/270] Running noise_50__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Frozen smoke-output equivalence: PASS | noise_50__seed_01
  Completed: noise_50__seed_01
  Raw flips: 2021337 | model-label changes: 103102 | dependent REC changes: 2575312
  Training failures: 103071 | condition seconds: 246.58

--------------------------------------------------------------------------------------------------------------
[worker 3/45 | global 2/270] Running noise_05__seed_01
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


...

  Fitting: NaiveBayes
  Completed: noise_40__seed_05
  Raw flips: 1614981 | model-label changes: 82211 | dependent REC changes: 2507756
  Training failures: 82600 | condition seconds: 290.21

--------------------------------------------------------------------------------------------------------------
[worker 45/45 | global 45/270] Running noise_50__seed_05
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_05
  Raw flips: 2019432 | model-label changes: 102682 | dependent REC changes: 2573067
  Training failures: 102723 | condition seconds: 280.47

Validating all 45 assigned conditions for seed_01_05.

Worker validation:


,Check,Expected,Actual,Pass
0,Worker assigned seeds,"[1, 2, 3, 4, 5]","[1, 2, 3, 4, 5]",True
1,Worker conditions,45,45,True
2,Worker noise levels,"[0, 5, 10, 15, 20, 25, 30, 40, 50]","[0, 5, 10, 15, 20, 25, 30, 40, 50]",True
3,Duplicate condition keys,0,0,True
4,Files per condition,8,[8],True
5,Worker raw files,360,360,True
6,Worker ranking rows,5939010,5939010,True
7,Worker build-metric rows,5355,5355,True
8,Worker project-run rows,315,315,True
9,Worker model fits,180,180,True



=== PROJECT 24 STEP 5A WORKER seed_01_05 RESULT ===
Project: SonarSource@sonarqube
Worker notebook: Thesis_p24_seed1-5
Worker code revision: P24_SEEDS_01_05_V1_FROZEN_SCHEMA_SMOKE_EQUIVALENT
Accelerated engine: PROJECT_24_FAST_DEPENDENT_REC_V1_SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE
Assigned seeds: [1, 2, 3, 4, 5]
Conditions: 45 / 45
ML fits: 180 / 180
Ranking rows: 5939010
Build-metric rows: 5355
Project-run rows: 315
Condition-audit rows: 45
Training-median rows: 6795
Worker raw files: 360
Worker raw bytes: 86952171
Worker raw root SHA-256: 53c1767a911a5d52561e2e12777f18d74ef8386c151a60fdf75e1488c2ab5705
Baseline invariance failures: 0
Master Step 5A write-guard failures: 0
Validation checks: 31
Failed checks: 0
Completed this invocation: 45
Skipped validated conditions: 0
Runtime seconds this invocation: 11062.1
Worker checkpoint: /content/drive/MyDrive/Thesis_Experiment/Notes/project_24_step5a_worker_seed_01_05_checkpoint.json
Worker checkpoint SHA-256: c6e8d76fe7

In [2]:
# ==================================================================================================
# PROJECT 24 — STEP 5A PARALLEL WORKER
# RESUME-SAFE CHECKPOINT-SCHEMA-COMPATIBLE ACCELERATED SEED-SHARD EXECUTION
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS IN ITS ASSIGNED NEW PARALLEL COLAB WORKER NOTEBOOK ONLY.
# DO NOT RUN THIS WORKER CELL IN THE MASTER NOTEBOOK.
#
# PURPOSE:
# - validate the frozen Project 24 Step 4B smoke-test checkpoint and every upstream contract;
# - validate the accelerated REC engine against the exact frozen Step 4B smoke outputs;
# - execute only this worker's assigned non-overlapping seed shard;
# - checkpoint each completed condition independently using atomic output files;
# - resume safely after a Colab disconnect by skipping only fully validated conditions;
# - fit the four frozen ML techniques and evaluate the three frozen baselines;
# - write ranked-test, build-metric, project-run, model-fit, median, and audit outputs;
# - freeze an independent worker checkpoint; the master Step 5A freeze happens only after all workers PASS.
#
# SAFETY:
# - no registry write;
# - no modification of Projects 1–23;
# - six total seed shards are executed in two waves of three Colab runtimes;
# - no prior-project condition-output access;
# - clean evaluation data remain immutable;
# - incomplete assigned-condition outputs are preserved in this worker's private quarantine before rerun.
# - master Step 5A aggregate/checkpoint/status files are write-protected by an explicit guard.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import gc
import hashlib
import json
import os
import platform
import shutil
import time
import warnings
import tarfile

from google.colab import drive

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.simplefilter("ignore", PerformanceWarning)

print("=" * 136)
print("=== PROJECT 24 STEP 5A PARALLEL WORKER: SEEDS 01-05 ===")
print("=" * 136)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

EXPECTED_STEP4A_STATUS = (
    "PASS_PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)
EXPECTED_STEP4B_STATUS = (
    "PASS_PROJECT_24_TWO_CONDITION_END_TO_END_SMOKE_TEST"
)
STEP5A_STATUS = (
    "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
)
CONDITION_STATUS = "PASS_FULL_CONDITION"

EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "ff9ed031d21d67bf218d5eab70bd8502ca39953adedbe840348f382d5599314e"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "3f5a0cc24260b251fd8cd43f7391ff7d3cc3d517e8f79a8ad443f4002cdc9ff2"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)
EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)
EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)
EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_REC_IMPLEMENTATION = (
    "PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_"
    "ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT"
)

EXPECTED_STEP3A_CODE_REVISION = (
    "PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER_SCHEMA_FIX"
)

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

# Worker-shard constants are injected into each generated worker file.
WORKER_SEED_START = 16
WORKER_SEED_END = 20
WORKER_TAG = "seed_16_20"
WORKER_NOTEBOOK_NAME = "Thesis_p24_seed16-20"
WORKER_CODE_REVISION = "P24_SEEDS_16_20_V1_FROZEN_SCHEMA_SMOKE_EQUIVALENT"
WORKER_STATUS = "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_16_20_COMPLETE"
WORKER_IS_SMOKE_OWNER = (WORKER_SEED_START <= 1 <= WORKER_SEED_END)
SHARED_EQUIVALENCE_WAIT_SECONDS = 60 * 60
SHARED_EQUIVALENCE_POLL_SECONDS = 10

EXPECTED_REGISTERED_PROJECTS = 23

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_ROWS = 224_550
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_FAILING_EVAL_BUILDS = 17
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_EVALUATION_BUILDS = 1_072
EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = EXPECTED_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_RNG_ROWS = 121_224_030
ACCELERATED_ENGINE_VERSION = "PROJECT_24_FAST_DEPENDENT_REC_V1_SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE"
SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))
WORKER_REPETITION_SEEDS = list(range(WORKER_SEED_START, WORKER_SEED_END + 1))
EXPECTED_WORKER_CONDITIONS = len(WORKER_REPETITION_SEEDS) * len(NOISE_LEVELS)
EXPECTED_WORKER_RAW_FILES = EXPECTED_WORKER_CONDITIONS * EXPECTED_FILES_PER_CONDITION
EXPECTED_WORKER_RANKING_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_RANKING_ROWS_PER_CONDITION
EXPECTED_WORKER_BUILD_METRIC_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION
EXPECTED_WORKER_PROJECT_RUN_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION
EXPECTED_WORKER_MODEL_FITS = EXPECTED_WORKER_CONDITIONS * EXPECTED_MODEL_FIT_ROWS_PER_CONDITION
EXPECTED_WORKER_TRAINING_MEDIAN_ROWS = EXPECTED_WORKER_CONDITIONS * EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION
RECENT_WINDOW = 6

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },
    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },
    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },
    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SOURCE_DIR = Path("/content/datasets/datasets/SonarSource@sonarqube")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_24_selection"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_24_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_24_fixed_chronological_builds.csv"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_24_selection_checkpoint.json"

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
BUILD_ENTITY_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
CLEAN_RECONSTRUCTED_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
CLEAN_ANCHOR_OFFSETS_PATH = REC_PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_24_rec_reconstruction_checkpoint.json"

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
RAW_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
MODEL_EVALUATION_COHORT_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
MODEL_RAW_TRAIN_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
MODEL_RAW_EVAL_LINK_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
RNG_MANIFEST_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
CONDITION_PLAN_PATH = NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
NOISE_PLAN_CHECKPOINT_PATH = NOTES_ROOT / "project_24_noise_plan_checkpoint.json"

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
PREDICTOR_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_predictor_contract.csv"
MODEL_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_model_contract.json"
BASELINE_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_baseline_contract.json"
RANKING_CONTRACT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_ranking_contract.json"
STEP4A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4a_status.json"
STEP4A_REPORT_PATH = RUNTIME_ROOT / f"{PROJECT_SHORT}_step4a_report.json"
RUNTIME_CHECKPOINT_PATH = NOTES_ROOT / "project_24_runtime_contract_checkpoint.json"

STEP4B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step4b_status.json"
SMOKE_CHECKPOINT_PATH = NOTES_ROOT / "project_24_smoke_test_checkpoint.json"
SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"

# Official master Step 5A outputs. Workers may READ their state for guards but MUST NOT write them.
CONDITION_INVENTORY_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
RAW_MANIFEST_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
BASELINE_INVARIANCE_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
COMBINED_CONDITION_AUDIT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
COMBINED_PROJECT_RUNS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
COMBINED_BUILD_METRICS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
COMBINED_MODEL_FITS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
STEP5A_VALIDATION_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
STEP5A_REPORT_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
MASTER_RUN_PROGRESS_PATH = FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_CHECKPOINT_PATH = NOTES_ROOT / "project_24_step5a_checkpoint.json"

# Shared accelerated-equivalence audit: ONLY the seed-1 worker writes it; other workers wait/read it.
ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT
    / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

# Private worker outputs: no path overlaps another worker.
PARALLEL_WORKERS_ROOT = FULL_EXPERIMENT_ROOT / "parallel_workers"
WORKER_ROOT = PARALLEL_WORKERS_ROOT / WORKER_TAG
INCOMPLETE_BACKUP_ROOT = WORKER_ROOT / "incomplete_condition_backups"
WORKER_INVENTORY_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_condition_inventory.csv"
WORKER_RAW_MANIFEST_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_raw_manifest.csv"
WORKER_BASELINE_INVARIANCE_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_baseline_invariance.csv"
WORKER_COMBINED_AUDIT_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_combined_condition_audit.csv"
WORKER_VALIDATION_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_validation.csv"
WORKER_REPORT_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_report.json"
RUN_PROGRESS_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_run_progress.json"
WORKER_STATUS_PATH = WORKER_ROOT / f"{PROJECT_SHORT}_{WORKER_TAG}_status.json"
WORKER_CHECKPOINT_PATH = NOTES_ROOT / f"project_24_step5a_worker_{WORKER_TAG}_checkpoint.json"

# --------------------------------------------------------------------------------------------------
# 2B. FRESH-WORKER RUNTIME BOOTSTRAP
# --------------------------------------------------------------------------------------------------

print("Worker notebook:", WORKER_NOTEBOOK_NAME)
print("Assigned repetition seeds:", WORKER_REPETITION_SEEDS)
print("Assigned conditions:", EXPECTED_WORKER_CONDITIONS)
print("Mounting the shared thesis Google Drive.")

drive.mount(
    "/content/drive",
    force_remount=False,
)

ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
REQUIRED_SOURCE_FILE_NAMES = {
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
}

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(
        "Frozen TCP-CI archive is missing from Google Drive:\n"
        f"{ARCHIVE_PATH}"
    )

source_ready = bool(
    SOURCE_DIR.is_dir()
    and REQUIRED_SOURCE_FILE_NAMES.issubset({
        path.name
        for path in SOURCE_DIR.iterdir()
        if path.is_file()
    })
)

if not source_ready:
    print("Restoring only SonarSource@sonarqube from the frozen TCP-CI archive into this worker runtime.")
    local_dataset_root = Path("/content/datasets")
    local_dataset_root.mkdir(parents=True, exist_ok=True)
    member_prefix = "datasets/SonarSource@sonarqube/"
    extracted_files = 0
    resolved_root = local_dataset_root.resolve()

    with tarfile.open(ARCHIVE_PATH, mode="r:gz") as archive:
        for member in archive:
            member_name = member.name.replace("\\", "/").lstrip("/")
            if not (
                member_name == "datasets/SonarSource@sonarqube"
                or member_name.startswith(member_prefix)
            ):
                continue

            target_path = local_dataset_root / member_name
            resolved_target = target_path.resolve()
            if resolved_target != resolved_root and resolved_root not in resolved_target.parents:
                raise RuntimeError(
                    "Unsafe archive member encountered during worker bootstrap:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                target_path.parent.mkdir(parents=True, exist_ok=True)
                source_handle = archive.extractfile(member)
                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )
                with source_handle, target_path.open("wb") as output_handle:
                    shutil.copyfileobj(source_handle, output_handle, length=8 * 1024 * 1024)
                extracted_files += 1

    print("Project source files restored:", extracted_files)

source_file_names = {
    path.name
    for path in SOURCE_DIR.iterdir()
    if path.is_file()
} if SOURCE_DIR.is_dir() else set()
missing_worker_source_files = sorted(REQUIRED_SOURCE_FILE_NAMES - source_file_names)
if missing_worker_source_files:
    raise FileNotFoundError(
        "Worker runtime could not restore the complete frozen SonarSource@sonarqube source:\n"
        + "\n".join(missing_worker_source_files)
    )

WORKER_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

def sha256_array(values, dtype):
    array = np.asarray(values).astype(dtype, copy=False)
    return hashlib.sha256(array.tobytes(order="C")).hexdigest()

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary_path, path)

def atomic_csv(path, frame, compression=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(temporary_path, path)

def atomic_parquet(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_parquet(
        temporary_path,
        index=False,
    )

    os.replace(temporary_path, path)

def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )
        digest.update(line.encode("utf-8"))

    return digest.hexdigest()

def parse_int(values, label):
    numeric = pd.to_numeric(values, errors="coerce")

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains {int(numeric.isna().sum())} missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")

def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })

def deterministic_seed(repetition_seed, stream_name):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(material).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="little",
        signed=False,
    )

def deterministic_random_build_seed(repetition_seed, build_id):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )

def create_models(repetition_seed):
    return {
        "RandomForest": RandomForestClassifier(
            **MODEL_CONFIG["RandomForest"],
            random_state=deterministic_seed(
                repetition_seed,
                "RandomForest_model",
            ),
        ),
        "XGBoost": XGBClassifier(
            **MODEL_CONFIG["XGBoost"],
            random_state=deterministic_seed(
                repetition_seed,
                "XGBoost_model",
            ),
        ),
        "LightGBM": LGBMClassifier(
            **MODEL_CONFIG["LightGBM"],
            random_state=deterministic_seed(
                repetition_seed,
                "LightGBM_model",
            ),
        ),
        "NaiveBayes": GaussianNB(
            **MODEL_CONFIG["NaiveBayes"]
        ),
    }

def calculate_apfd(failures):
    failures = np.asarray(failures, dtype=np.int8)
    number_of_tests = len(failures)
    number_of_failures = int(failures.sum())

    if number_of_tests == 0 or number_of_failures == 0:
        return np.nan

    failure_positions = np.flatnonzero(failures == 1) + 1

    return float(
        1.0
        - (
            failure_positions.sum()
            / (number_of_tests * number_of_failures)
        )
        + (1.0 / (2.0 * number_of_tests))
    )

def calculate_apfdc(failures, durations):
    failures = np.asarray(failures, dtype=np.int8)
    durations = np.asarray(durations, dtype=float)

    if len(failures) != len(durations):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if len(failures) == 0 or failures.sum() == 0:
        return np.nan

    if not np.isfinite(durations).all():
        raise ValueError(
            "Durations contain missing or infinite values."
        )

    if (durations < 0).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(durations.sum())

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([0.0]),
        np.cumsum(durations)[:-1],
    ])

    failure_mask = failures == 1
    midpoint_detection_times = (
        cumulative_before[failure_mask]
        + (0.5 * durations[failure_mask])
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )

def calculate_rates(history):
    history_length = len(history)

    if history_length == 0:
        raise ValueError(
            "Rate calculation requires non-empty history."
        )

    verdicts = history["verdict"]

    return (
        float(verdicts.ne(0).sum() / history_length),
        float(verdicts.eq(2).sum() / history_length),
        float(verdicts.eq(1).sum() / history_length),
        float(history["transition"].eq(1).sum() / history_length),
    )

def calculate_max_test_file_rate(
    history,
    target_column,
    current_changed_entities,
    entity_changed_builds,
):
    target_builds = (
        history.loc[
            history[target_column].gt(0),
            "build",
        ]
        .drop_duplicates()
        .astype(int)
        .tolist()
    )

    if len(target_builds) == 0:
        return -1.0

    target_build_set = set(target_builds)
    maximum_frequency = 0

    for entity_id in current_changed_entities:
        changed_builds = entity_changed_builds.get(
            int(entity_id),
            set(),
        )

        overlap_count = len(
            changed_builds.intersection(target_build_set)
        )

        maximum_frequency = max(
            maximum_frequency,
            overlap_count,
        )

    if maximum_frequency == 0:
        return 0.0

    return float(
        maximum_frequency
        / len(target_builds)
    )

def reconstruct_rec_features(
    execution_history,
    requested_rows,
    global_build_position,
    changed_entities_by_build,
    entity_changed_builds,
    recent_window=6,
):
    requested_pairs = set(
        zip(
            requested_rows["Build"].astype(int),
            requested_rows["Test"].astype(int),
        )
    )

    reconstructed_records = []
    test_groups = execution_history.groupby(
        "test",
        sort=False,
    )
    total_tests = int(
        execution_history["test"].nunique()
    )

    for test_index, (test_id, test_history) in enumerate(
        test_groups,
        start=1,
    ):
        test_history = (
            test_history.sort_values(
                [
                    "build_order",
                    "job",
                ],
                kind="mergesort",
            )
            .reset_index(drop=True)
            .copy()
        )

        test_history["transition"] = (
            test_history["verdict"]
            .diff()
            .fillna(0)
            .ne(0)
            .astype(int)
        )

        first_test_build = int(
            test_history.iloc[0]["build"]
        )

        for current_position in range(len(test_history)):
            current_row = test_history.iloc[current_position]
            current_build = int(current_row["build"])
            current_test = int(test_id)
            pair = (current_build, current_test)

            if pair not in requested_pairs:
                continue

            history = (
                test_history.iloc[:current_position]
                .copy()
                .reset_index(drop=True)
            )

            record = {
                "Build": current_build,
                "Test": current_test,
            }

            if history.empty:
                for feature in REC_FEATURES:
                    record[feature] = -1.0

                record["REC_Age"] = 0.0
                reconstructed_records.append(record)
                continue

            recent_history = history.tail(recent_window).copy()

            age = float(
                global_build_position[current_build]
                - global_build_position[first_test_build]
            )

            failure_positions = np.flatnonzero(
                history["verdict"].to_numpy() > 0
            )

            last_failure_age = (
                -1.0
                if len(failure_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(failure_positions[-1])
                )
            )

            transition_positions = np.flatnonzero(
                history["transition"].to_numpy() > 0
            )

            last_transition_age = (
                -1.0
                if len(transition_positions) == 0
                else float(
                    len(history)
                    - 1
                    - int(transition_positions[-1])
                )
            )

            (
                recent_fail_rate,
                recent_assert_rate,
                recent_exc_rate,
                recent_transition_rate,
            ) = calculate_rates(recent_history)

            (
                total_fail_rate,
                total_assert_rate,
                total_exc_rate,
                total_transition_rate,
            ) = calculate_rates(history)

            current_changed_entities = (
                changed_entities_by_build.get(
                    current_build,
                    set(),
                )
            )

            max_file_fail_rate = calculate_max_test_file_rate(
                history=history,
                target_column="verdict",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            max_file_transition_rate = calculate_max_test_file_rate(
                history=history,
                target_column="transition",
                current_changed_entities=current_changed_entities,
                entity_changed_builds=entity_changed_builds,
            )

            record.update({
                "REC_Age": age,
                "REC_LastFailureAge": last_failure_age,
                "REC_LastTransitionAge": last_transition_age,
                "REC_RecentAvgExeTime": float(
                    recent_history["duration"].mean()
                ),
                "REC_RecentMaxExeTime": float(
                    recent_history["duration"].max()
                ),
                "REC_RecentFailRate": recent_fail_rate,
                "REC_RecentAssertRate": recent_assert_rate,
                "REC_RecentExcRate": recent_exc_rate,
                "REC_RecentTransitionRate": recent_transition_rate,
                "REC_TotalAvgExeTime": float(
                    history["duration"].mean()
                ),
                "REC_TotalMaxExeTime": float(
                    history["duration"].max()
                ),
                "REC_TotalFailRate": total_fail_rate,
                "REC_TotalAssertRate": total_assert_rate,
                "REC_TotalExcRate": total_exc_rate,
                "REC_TotalTransitionRate": total_transition_rate,
                "REC_LastVerdict": float(
                    recent_history.iloc[-1]["verdict"]
                ),
                "REC_LastExeTime": float(
                    recent_history.iloc[-1]["duration"]
                ),
                "REC_MaxTestFileFailRate": max_file_fail_rate,
                "REC_MaxTestFileTransitionRate": (
                    max_file_transition_rate
                ),
            })

            reconstructed_records.append(record)

        if test_index % 100 == 0 or test_index == total_tests:
            print(
                "    REC reconstruction progress:",
                test_index,
                "/",
                total_tests,
                "tests | reconstructed rows:",
                len(reconstructed_records),
            )

    return pd.DataFrame(reconstructed_records)

def reconstruct_dependent_rec_fast(condition_combined_verdict):
    """
    Reconstruct only the 13 verdict-dependent REC features.

    This is algebraically equivalent to the frozen Step 4B implementation:
    - the exact Step 2B-frozen InferredTestOrder is used;
    - only prior executions contribute to each current row;
    - recent window = 6;
    - verdict 2 = assertion, verdict 1 = exception;
    - file-history rates use distinct prior target builds and current-build entities;
    - builds with no mapped entities produce 0 when target history exists and -1 when it does not.
    """
    condition_combined_verdict = np.asarray(
        condition_combined_verdict,
        dtype=np.int16,
    )

    if len(condition_combined_verdict) != EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS:
        raise RuntimeError(
            "Accelerated REC engine received the wrong execution-history length."
        )

    verdict_sorted = condition_combined_verdict[
        accelerated_history_combined_indices
    ]

    result = np.full(
        (
            EXPECTED_MODEL_ROWS,
            len(VERDICT_DEPENDENT_REC),
        ),
        -1.0,
        dtype=np.float64,
    )

    for group_index in range(accelerated_group_count):
        requested_model_indices = accelerated_requested_model_indices[group_index]

        if len(requested_model_indices) == 0:
            continue

        start = int(accelerated_group_starts[group_index])
        end = int(accelerated_group_ends[group_index])
        local_positions = accelerated_requested_local_positions[group_index]

        verdict = verdict_sorted[start:end]
        group_length = len(verdict)
        position = np.arange(group_length, dtype=np.int64)

        failure = verdict > 0
        assertion = verdict == 2
        exception = verdict == 1
        transition = np.zeros(group_length, dtype=np.bool_)

        if group_length > 1:
            transition[1:] = verdict[1:] != verdict[:-1]

        failure_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(failure, dtype=np.int64),
        ))
        assertion_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(assertion, dtype=np.int64),
        ))
        exception_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(exception, dtype=np.int64),
        ))
        transition_prefix = np.concatenate((
            np.array([0], dtype=np.int64),
            np.cumsum(transition, dtype=np.int64),
        ))

        has_history = local_positions > 0

        if has_history.any():
            requested_with_history = np.flatnonzero(has_history)
            current_positions = local_positions[requested_with_history]
            model_indices = requested_model_indices[requested_with_history]

            recent_starts = np.maximum(
                0,
                current_positions - RECENT_WINDOW,
            )
            recent_lengths = current_positions - recent_starts

            last_failure_position = np.maximum.accumulate(
                np.where(failure, position, -1)
            )
            last_transition_position = np.maximum.accumulate(
                np.where(transition, position, -1)
            )

            prior_last_failure = last_failure_position[
                current_positions - 1
            ]
            prior_last_transition = last_transition_position[
                current_positions - 1
            ]

            result[model_indices, 0] = np.where(
                prior_last_failure >= 0,
                current_positions - 1 - prior_last_failure,
                -1,
            ).astype(np.float64)
            result[model_indices, 1] = np.where(
                prior_last_transition >= 0,
                current_positions - 1 - prior_last_transition,
                -1,
            ).astype(np.float64)

            result[model_indices, 2] = (
                failure_prefix[current_positions]
                - failure_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 3] = (
                assertion_prefix[current_positions]
                - assertion_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 4] = (
                exception_prefix[current_positions]
                - exception_prefix[recent_starts]
            ) / recent_lengths
            result[model_indices, 5] = (
                transition_prefix[current_positions]
                - transition_prefix[recent_starts]
            ) / recent_lengths

            result[model_indices, 6] = (
                failure_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 7] = (
                assertion_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 8] = (
                exception_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 9] = (
                transition_prefix[current_positions]
                / current_positions
            )
            result[model_indices, 10] = verdict[
                current_positions - 1
            ].astype(np.float64)

        # File-history features. The counters contain only target executions
        # strictly before the current position, matching Step 4B exactly.
        failure_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        transition_entity_counts = np.zeros(
            accelerated_entity_count,
            dtype=np.int32,
        )
        failure_denominator = 0
        transition_denominator = 0
        requested_pointer = 0

        group_build_indices = accelerated_history_build_dense_indices[
            start:end
        ]

        for local_position in range(group_length):
            while (
                requested_pointer < len(local_positions)
                and int(local_positions[requested_pointer]) == local_position
            ):
                model_index = int(
                    requested_model_indices[requested_pointer]
                )

                if local_position > 0:
                    current_entities = accelerated_build_entity_arrays[
                        int(group_build_indices[local_position])
                    ]

                    if failure_denominator == 0:
                        result[model_index, 11] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 11] = 0.0
                    else:
                        result[model_index, 11] = float(
                            failure_entity_counts[
                                current_entities
                            ].max()
                            / failure_denominator
                        )

                    if transition_denominator == 0:
                        result[model_index, 12] = -1.0
                    elif len(current_entities) == 0:
                        result[model_index, 12] = 0.0
                    else:
                        result[model_index, 12] = float(
                            transition_entity_counts[
                                current_entities
                            ].max()
                            / transition_denominator
                        )

                requested_pointer += 1

            changed_entities = accelerated_build_entity_arrays[
                int(group_build_indices[local_position])
            ]

            if failure[local_position]:
                if len(changed_entities) != 0:
                    failure_entity_counts[changed_entities] += 1
                failure_denominator += 1

            if transition[local_position]:
                if len(changed_entities) != 0:
                    transition_entity_counts[changed_entities] += 1
                transition_denominator += 1

        if requested_pointer != len(local_positions):
            raise RuntimeError(
                "Accelerated REC engine did not emit every requested row."
            )

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Accelerated REC engine produced non-finite values."
        )

    return result

def maximum_absolute_difference(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)

    if left.shape != right.shape:
        return np.inf

    if left.size == 0:
        return 0.0

    return float(np.max(np.abs(left - right)))

def compare_full_condition_to_smoke(condition_key, full_condition_dir):
    """Compare all scientific outputs with the frozen Step 4B condition."""
    full_condition_dir = Path(full_condition_dir)
    smoke_condition_dir = SMOKE_ROOT / condition_key

    required_names = [
        "rankings.csv.gz",
        "build_metrics.csv",
        "project_runs.csv",
        "model_fits.csv",
        "training_medians.csv",
        "condition_audit.csv",
    ]

    for name in required_names:
        if not (full_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Accelerated equivalence input missing: {full_condition_dir / name}"
            )
        if not (smoke_condition_dir / name).is_file():
            raise FileNotFoundError(
                f"Frozen smoke output missing: {smoke_condition_dir / name}"
            )

    actual_rankings = pd.read_csv(
        full_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_rankings = pd.read_csv(
        smoke_condition_dir / "rankings.csv.gz",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build", "Test"],
        kind="mergesort",
    ).reset_index(drop=True)

    ranking_key_columns = [
        "Technique",
        "Build",
        "Test",
        "Rank",
        "CleanVerdict",
        "CleanFailure",
    ]
    ranking_keys_equal = bool(
        len(actual_rankings) == len(smoke_rankings)
        and actual_rankings[ranking_key_columns].equals(
            smoke_rankings[ranking_key_columns]
        )
    )
    ranking_score_max_difference = maximum_absolute_difference(
        actual_rankings["Score"].to_numpy(dtype=float),
        smoke_rankings["Score"].to_numpy(dtype=float),
    )

    actual_build = pd.read_csv(
        full_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    smoke_build = pd.read_csv(
        smoke_condition_dir / "build_metrics.csv",
        low_memory=False,
    ).sort_values(
        ["Technique", "Build"],
        kind="mergesort",
    ).reset_index(drop=True)
    build_keys_equal = bool(
        len(actual_build) == len(smoke_build)
        and actual_build[["Technique", "Build", "Tests", "Failures"]].equals(
            smoke_build[["Technique", "Build", "Tests", "Failures"]]
        )
    )
    build_metric_max_difference = maximum_absolute_difference(
        actual_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
        smoke_build[["TotalDuration", "APFDc", "APFD"]].to_numpy(dtype=float),
    )

    actual_project = pd.read_csv(
        full_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_project = pd.read_csv(
        smoke_condition_dir / "project_runs.csv",
        low_memory=False,
    ).sort_values("Technique", kind="mergesort").reset_index(drop=True)
    project_keys_equal = bool(
        len(actual_project) == len(smoke_project)
        and actual_project[[
            "Technique",
            "EvaluationBuilds",
            "ScoredFailingBuilds",
            "EvaluationRows",
            "EvaluationFailures",
        ]].equals(
            smoke_project[[
                "Technique",
                "EvaluationBuilds",
                "ScoredFailingBuilds",
                "EvaluationRows",
                "EvaluationFailures",
            ]]
        )
    )
    project_metric_max_difference = maximum_absolute_difference(
        actual_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
        smoke_project[[
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ]].to_numpy(dtype=float),
    )

    actual_medians = pd.read_csv(
        full_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    smoke_medians = pd.read_csv(
        smoke_condition_dir / "training_medians.csv",
        low_memory=False,
    ).sort_values("PredictorOrder", kind="mergesort").reset_index(drop=True)
    median_keys_equal = bool(
        len(actual_medians) == len(smoke_medians)
        and actual_medians[["PredictorOrder", "Predictor"]].equals(
            smoke_medians[["PredictorOrder", "Predictor"]]
        )
    )
    median_max_difference = maximum_absolute_difference(
        actual_medians["TrainingMedian"].to_numpy(dtype=float),
        smoke_medians["TrainingMedian"].to_numpy(dtype=float),
    )

    actual_fits = pd.read_csv(
        full_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    smoke_fits = pd.read_csv(
        smoke_condition_dir / "model_fits.csv",
        low_memory=False,
    ).fillna("").sort_values("Technique", kind="mergesort").reset_index(drop=True)
    fit_contract_columns = [
        "Technique",
        "TrainingRows",
        "TrainingFailures",
        "Predictors",
        "ClassesJSON",
        "Status",
        "Error",
    ]
    fit_contract_equal = bool(
        len(actual_fits) == len(smoke_fits)
        and actual_fits[fit_contract_columns].equals(
            smoke_fits[fit_contract_columns]
        )
    )

    actual_audit = pd.read_csv(
        full_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    smoke_audit = pd.read_csv(
        smoke_condition_dir / "condition_audit.csv",
        low_memory=False,
    ).iloc[0]
    # Project 24 Step 4B froze the worker-compatible condition-audit schema used here.
    # The Project 23 worker precedent used a larger audit schema, so comparing
    # those predecessor-only metadata columns against Project 24's smoke output
    # raises KeyError (for example RawTrainingRows).
    #
    # Smoke equivalence must compare the scientific audit fields that were
    # actually frozen by Project 24 Step 4B. Extra full-condition metadata are
    # validated separately by the worker's own condition-completion checks.
    audit_columns = [
        "ProjectNumber",
        "Project",
        "ProjectSlug",
        "ConditionKey",
        "NoisePercent",
        "RepetitionSeed",
        "NumberFlipped",
        "ModelLabelChanges",
        "TrainingFailures",
        "DependentRECChanges",
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
        "RankingRows",
        "BuildMetricRows",
        "ProjectRunRows",
        "TrainingMedianRows",
    ]

    missing_actual_audit_columns = [
        column
        for column in audit_columns
        if column not in actual_audit.index
    ]
    missing_smoke_audit_columns = [
        column
        for column in audit_columns
        if column not in smoke_audit.index
    ]

    if missing_actual_audit_columns or missing_smoke_audit_columns:
        raise RuntimeError(
            "Project 24 smoke-equivalence audit schema differs. "
            f"Missing from full condition={missing_actual_audit_columns}; "
            f"missing from frozen smoke={missing_smoke_audit_columns}; "
            f"full columns={actual_audit.index.tolist()}; "
            f"smoke columns={smoke_audit.index.tolist()}"
        )

    audit_equal = bool(
        all(
            str(actual_audit[column]) == str(smoke_audit[column])
            for column in audit_columns
        )
    )

    passed = bool(
        ranking_keys_equal
        and ranking_score_max_difference <= 1e-12
        and build_keys_equal
        and build_metric_max_difference <= 1e-12
        and project_keys_equal
        and project_metric_max_difference <= 1e-12
        and median_keys_equal
        and median_max_difference <= 1e-12
        and fit_contract_equal
        and audit_equal
    )

    return {
        "ConditionKey": condition_key,
        "EngineVersion": ACCELERATED_ENGINE_VERSION,
        "RankingKeysEqual": ranking_keys_equal,
        "RankingScoreMaxDifference": ranking_score_max_difference,
        "BuildMetricKeysEqual": build_keys_equal,
        "BuildMetricMaxDifference": build_metric_max_difference,
        "ProjectRunKeysEqual": project_keys_equal,
        "ProjectMetricMaxDifference": project_metric_max_difference,
        "TrainingMedianKeysEqual": median_keys_equal,
        "TrainingMedianMaxDifference": median_max_difference,
        "ModelFitContractEqual": fit_contract_equal,
        "ConditionAuditEqual": audit_equal,
        "Pass": passed,
    }

def positive_probability(estimator, matrix):
    probabilities = estimator.predict_proba(matrix)
    classes = np.asarray(estimator.classes_)
    positive_columns = np.flatnonzero(classes == 1)

    if len(positive_columns) != 1:
        raise RuntimeError(
            "Fitted estimator does not expose exactly one class-1 probability column."
        )

    scores = probabilities[:, int(positive_columns[0])]

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Model produced non-finite failure probabilities."
        )

    if ((scores < 0) | (scores > 1)).any():
        raise RuntimeError(
            "Model produced probabilities outside [0,1]."
        )

    return scores.astype(float, copy=False)

def make_ranking(
    evaluation_meta,
    technique,
    scores,
    ascending_score,
):
    ranking = evaluation_meta.copy()
    ranking["Technique"] = technique
    ranking["Score"] = np.asarray(scores, dtype=float)

    if len(ranking) != EXPECTED_MODEL_EVAL_ROWS:
        raise RuntimeError(
            f"{technique} ranking input has the wrong row count."
        )

    if not np.isfinite(ranking["Score"].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f"{technique} ranking contains non-finite scores."
        )

    ranking = (
        ranking.sort_values(
            [
                "Build",
                "Score",
                "Test",
            ],
            ascending=[
                True,
                bool(ascending_score),
                True,
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranking["Rank"] = (
        ranking.groupby(
            "Build",
            sort=False,
        )
        .cumcount()
        .add(1)
        .astype("int64")
    )

    return ranking[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
            "Build",
            "Test",
            "Rank",
            "Score",
            "CleanVerdict",
            "CleanFailure",
            "Duration",
        ]
    ]

def calculate_condition_metrics(rankings):
    build_metric_records = []

    failing_rankings = rankings.loc[
        rankings["Build"].isin(failing_evaluation_builds)
    ].copy()

    for (technique, build_id), build_ranking in failing_rankings.groupby(
        [
            "Technique",
            "Build",
        ],
        sort=False,
    ):
        build_ranking = build_ranking.sort_values(
            "Rank",
            kind="mergesort",
        )

        failures = build_ranking[
            "CleanFailure"
        ].to_numpy(dtype=np.int8)

        durations = build_ranking[
            "Duration"
        ].to_numpy(dtype=float)

        number_of_failures = int(failures.sum())

        if number_of_failures <= 0:
            raise RuntimeError(
                "A supposedly failing evaluation build has no failures."
            )

        apfd = calculate_apfd(failures)
        apfdc = calculate_apfdc(failures, durations)

        build_metric_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                build_ranking["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                build_ranking["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                build_ranking["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "Build": int(build_id),
            "Tests": int(len(build_ranking)),
            "Failures": number_of_failures,
            "TotalDuration": float(durations.sum()),
            "APFDc": float(apfdc),
            "APFD": float(apfd),
        })

    build_metrics = pd.DataFrame(build_metric_records)

    project_run_records = []

    for technique, technique_metrics in build_metrics.groupby(
        "Technique",
        sort=False,
    ):
        project_run_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": str(
                technique_metrics["ConditionKey"].iloc[0]
            ),
            "NoisePercent": int(
                technique_metrics["NoisePercent"].iloc[0]
            ),
            "RepetitionSeed": int(
                technique_metrics["RepetitionSeed"].iloc[0]
            ),
            "Technique": technique,
            "EvaluationBuilds": EXPECTED_EVALUATION_BUILDS,
            "ScoredFailingBuilds": int(len(technique_metrics)),
            "EvaluationRows": EXPECTED_MODEL_EVAL_ROWS,
            "EvaluationFailures": EXPECTED_MODEL_EVAL_FAILURES,
            "MeanAPFDc": float(
                technique_metrics["APFDc"].mean()
            ),
            "MedianAPFDc": float(
                technique_metrics["APFDc"].median()
            ),
            "MeanAPFD": float(
                technique_metrics["APFD"].mean()
            ),
            "MedianAPFD": float(
                technique_metrics["APFD"].median()
            ),
        })

    project_runs = pd.DataFrame(project_run_records)

    return build_metrics, project_runs

DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]

def directory_manifest(root):
    root = Path(root)
    rows = []

    if root.exists():
        for path in sorted(
            [
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ],
            key=lambda candidate: candidate.relative_to(root).as_posix(),
        ):
            rows.append({
                "RelativePath": path.relative_to(root).as_posix(),
                "Bytes": int(path.stat().st_size),
                "SHA256": sha256_file(path),
            })

    return pd.DataFrame(
        rows,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )

def directory_root_hash(manifest):
    if manifest is None:
        raise TypeError(
            "Directory manifest cannot be None."
        )

    missing_columns = [
        column
        for column in DIRECTORY_MANIFEST_COLUMNS
        if column not in manifest.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Directory manifest is missing required columns: "
            + ", ".join(missing_columns)
        )

    digest = hashlib.sha256()

    if manifest.empty:
        return digest.hexdigest()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()

def path_guard_state(path):
    path = Path(path)
    if not path.exists():
        return {"Exists": False, "Bytes": 0, "SHA256": ""}
    if not path.is_file():
        raise RuntimeError(f"Master write-guard path is unexpectedly not a file: {path}")
    return {
        "Exists": True,
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }


def load_valid_shared_equivalence():
    if not ACCELERATED_EQUIVALENCE_PATH.is_file():
        return None
    try:
        frame = pd.read_csv(ACCELERATED_EQUIVALENCE_PATH, low_memory=False)
    except Exception:
        return None
    required_columns = {"ConditionKey", "Pass"}
    if not required_columns.issubset(frame.columns):
        return None
    if len(frame) != len(SMOKE_EQUIVALENCE_KEYS):
        return None
    if sorted(frame["ConditionKey"].astype(str).tolist()) != sorted(SMOKE_EQUIVALENCE_KEYS):
        return None
    pass_values = frame["Pass"]
    if pass_values.dtype == bool:
        passed = pass_values
    else:
        passed = pass_values.astype(str).str.strip().str.lower().map({"true": True, "false": False})
        if passed.isna().any():
            return None
    if not bool(passed.all()):
        return None
    return frame


def wait_for_shared_equivalence():
    deadline = time.time() + SHARED_EQUIVALENCE_WAIT_SECONDS
    announced = False
    while time.time() < deadline:
        frame = load_valid_shared_equivalence()
        if frame is not None:
            return frame
        if not announced:
            print(
                "Waiting for seed-1 worker to freeze the two accelerated smoke-equivalence checks..."
            )
            announced = True
        time.sleep(SHARED_EQUIVALENCE_POLL_SECONDS)
    raise TimeoutError(
        "Timed out waiting for the seed-1 worker accelerated-equivalence audit. "
        "Check Worker A output before retrying this worker cell."
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN CHECKPOINTS
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SOURCE_DIR / "builds.csv",
    SOURCE_DIR / "contributors.csv",
    SOURCE_DIR / "dataset.csv",
    SOURCE_DIR / "entity_change_history.csv",
    SOURCE_DIR / "exe.csv",
    SOURCE_DIR / "id_map.csv",
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SELECTION_CHECKPOINT_PATH,
    BUILD_ENTITY_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    REC_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    PREDICTOR_CONTRACT_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_STATUS_PATH,
    STEP4A_REPORT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    STEP4B_STATUS_PATH,
    SMOKE_CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 24 Step 5A inputs are missing:\n"
        + "\n".join(missing_paths)
    )

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint_sha256 = sha256_file(
    SMOKE_CHECKPOINT_PATH
)

if selection_checkpoint_sha256 != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 selection checkpoint SHA-256 differs."
    )

if rec_checkpoint_sha256 != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 REC checkpoint SHA-256 differs."
    )

if noise_plan_checkpoint_sha256 != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 noise-plan checkpoint SHA-256 differs."
    )

if runtime_checkpoint_sha256 != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 runtime-contract checkpoint SHA-256 differs."
    )

if smoke_checkpoint_sha256 != EXPECTED_SMOKE_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 smoke-test checkpoint SHA-256 differs."
    )

runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
step4a_status = load_json(
    STEP4A_STATUS_PATH
)
step4a_report = load_json(
    STEP4A_REPORT_PATH
)
step4b_status = load_json(
    STEP4B_STATUS_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload in [
    ("runtime checkpoint", runtime_checkpoint),
    ("Step 4A status", step4a_status),
    ("Step 4A report", step4a_report),
]:
    if payload.get("Status") != EXPECTED_STEP4A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Step 4A PASS status."
        )

if runtime_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Runtime checkpoint project identity differs."
    )

if runtime_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Runtime checkpoint project slug differs."
    )

if runtime_checkpoint.get("SourceRootSHA256") != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint runtime-priority rule differs."
    )

if step4b_status.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 24 Step 4B status is not frozen successfully."
    )

if smoke_checkpoint.get("Status") != EXPECTED_STEP4B_STATUS:
    raise RuntimeError(
        "Project 24 smoke-test checkpoint is not frozen successfully."
    )

if smoke_checkpoint.get("Project") != PROJECT_NAME:
    raise RuntimeError(
        "Smoke-test checkpoint project identity differs."
    )

if smoke_checkpoint.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError(
        "Smoke-test checkpoint project slug differs."
    )

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_plan_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

if rec_checkpoint.get("ImplementationVersion") != EXPECTED_REC_IMPLEMENTATION:
    raise RuntimeError(
        "Frozen Project 24 REC implementation differs."
    )

if noise_plan_checkpoint.get("Step3ACodeRevision") != EXPECTED_STEP3A_CODE_REVISION:
    raise RuntimeError(
        "Frozen Project 24 Step 3A code revision differs."
    )

if runtime_checkpoint.get("Step3ACodeRevision") != EXPECTED_STEP3A_CODE_REVISION:
    raise RuntimeError(
        "Runtime checkpoint Step 3A revision linkage differs."
    )

if smoke_checkpoint.get("RECImplementation") != EXPECTED_REC_IMPLEMENTATION:
    raise RuntimeError(
        "Smoke checkpoint REC implementation linkage differs."
    )

if smoke_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Smoke-test checkpoint active reservations differ."
    )

# Step 4B validates the runtime-priority rule against the frozen Step 4A
# runtime checkpoint, but its checkpoint schema does not duplicate that field.
# Therefore, validate the frozen linkage instead of requiring an absent key.
if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke-test checkpoint does not link to the frozen runtime contract."
    )

if not bool(smoke_checkpoint.get("ReadyForFull270ConditionExperiment", False)):
    raise RuntimeError(
        "Smoke-test checkpoint does not authorise the full experiment."
    )

smoke_output_manifest = smoke_checkpoint.get("OutputManifest", [])
if not isinstance(smoke_output_manifest, list) or not smoke_output_manifest:
    raise RuntimeError(
        "Smoke-test checkpoint does not contain an output manifest."
    )

smoke_output_manifest_failures = 0
for item in smoke_output_manifest:
    output_path = Path(item["Path"])
    if (
        not output_path.is_file()
        or int(output_path.stat().st_size) != int(item["Bytes"])
        or sha256_file(output_path) != str(item["SHA256"])
    ):
        smoke_output_manifest_failures += 1

if smoke_output_manifest_failures != 0:
    raise RuntimeError(
        "One or more frozen Step 4B smoke outputs changed."
    )

# --------------------------------------------------------------------------------------------------
# 5. VERIFY SOURCE ROOT, REGISTRY, AND STEP 4A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

actual_runtime_versions = {
    "Python": platform.python_version(),
    "numpy": metadata.version("numpy"),
    "pandas": metadata.version("pandas"),
    "scikit-learn": metadata.version("scikit-learn"),
    "xgboost": metadata.version("xgboost"),
    "lightgbm": metadata.version("lightgbm"),
    "pyarrow": metadata.version("pyarrow"),
}

runtime_version_mismatches = {
    key: {
        "Expected": EXPECTED_RUNTIME_VERSIONS[key],
        "Actual": actual_runtime_versions[key],
    }
    for key in EXPECTED_RUNTIME_VERSIONS
    if actual_runtime_versions[key] != EXPECTED_RUNTIME_VERSIONS[key]
}

if runtime_version_mismatches:
    raise RuntimeError(
        "Fresh worker runtime does not match the frozen Step 4A environment:\n"
        + json.dumps(runtime_version_mismatches, indent=2, sort_keys=True)
    )

print("Worker runtime versions match frozen Step 4A exactly.")

registry_sha256_before = sha256_file(REGISTRY_PATH)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

project_number_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "projectnumber",
            "project_number",
            "project no",
            "projectno",
        }
    ),
    None,
)

project_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "project",
            "projectname",
            "project_name",
        }
    ),
    None,
)

status_column = next(
    (
        column
        for column in registry.columns
        if str(column).strip().lower()
        in {
            "status",
            "projectstatus",
            "project_status",
        }
    ),
    None,
)

if (
    project_number_column is None
    or project_column is None
    or status_column is None
):
    raise RuntimeError(
        "Could not resolve ProjectNumber, Project, and Status "
        "columns in the completion registry."
    )

registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–23."
    )

if not registry[
    status_column
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or str(
            matching_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 24 is already present in the completion registry."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_rows = []

for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 24 source file is missing: {source_path}"
        )

    current_source_rows.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })

current_source_manifest = pd.DataFrame(current_source_rows)
current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

if current_source_root_sha256 != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 source root differs before Step 5A."
    )

runtime_output_manifest = runtime_checkpoint.get(
    "RuntimeOutputManifest",
    [],
)

if not isinstance(runtime_output_manifest, list) or not runtime_output_manifest:
    raise RuntimeError(
        "Runtime checkpoint has no output manifest."
    )

runtime_manifest_records = []

for item in runtime_output_manifest:
    path = Path(item["Path"])
    expected_bytes = int(item["Bytes"])
    expected_sha256 = str(item["SHA256"]).lower()
    exists = path.is_file()
    actual_bytes = int(path.stat().st_size) if exists else -1
    actual_sha256 = sha256_file(path) if exists else "MISSING"
    passed = (
        exists
        and actual_bytes == expected_bytes
        and actual_sha256 == expected_sha256
    )

    runtime_manifest_records.append({
        "Path": str(path),
        "ExpectedBytes": expected_bytes,
        "ActualBytes": actual_bytes,
        "ExpectedSHA256": expected_sha256,
        "ActualSHA256": actual_sha256,
        "Pass": passed,
    })

runtime_manifest_audit = pd.DataFrame(
    runtime_manifest_records
)
runtime_manifest_failures = int(
    (~runtime_manifest_audit["Pass"]).sum()
)

if runtime_manifest_failures != 0:
    print("\nFailed Step 4A output-manifest checks:")
    display(
        runtime_manifest_audit.loc[
            ~runtime_manifest_audit["Pass"]
        ]
    )
    raise RuntimeError(
        "Step 4A output manifest no longer validates."
    )

full_raw_result_root_existed_before = FULL_RAW_RESULT_ROOT.exists()
full_raw_result_manifest_before = directory_manifest(
    FULL_RAW_RESULT_ROOT
)
full_raw_result_root_hash_before = directory_root_hash(
    full_raw_result_manifest_before
)

# --------------------------------------------------------------------------------------------------
# 6. LOAD FROZEN COHORTS, LINKS, CONDITION PLAN, AND RNG STREAM
# --------------------------------------------------------------------------------------------------

print("\nLoading frozen Project 24 cohorts and contracts.")

raw_training = pd.read_parquet(
    RAW_TRAINING_COHORT_PATH
)
raw_evaluation = pd.read_parquet(
    RAW_EVALUATION_COHORT_PATH
)
model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)
model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)
model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)
model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)
condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)
predictor_contract = pd.read_csv(
    PREDICTOR_CONTRACT_PATH,
    low_memory=False,
)
chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)
build_entity = pd.read_csv(
    BUILD_ENTITY_PATH,
    compression="gzip",
    low_memory=False,
)
anchor_offsets = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)
clean_reconstructed = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)

if len(raw_training) != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError("Raw training cohort row count differs.")
if len(raw_evaluation) != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError("Raw evaluation cohort row count differs.")
if len(model_training) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model training cohort row count differs.")
if len(model_evaluation) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model evaluation cohort row count differs.")
if len(model_train_link) != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError("Model/raw training link row count differs.")
if len(model_eval_link) != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError("Model/raw evaluation link row count differs.")
if len(anchor_offsets) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean anchor-offset row count differs.")
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean reconstructed REC row count differs.")

required_cohort_columns = {
    "Build",
    "Test",
    "Verdict",
}

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    missing = required_cohort_columns - set(frame.columns)
    if missing:
        raise RuntimeError(
            f"{label} cohort is missing columns: {sorted(missing)}"
        )

for label, frame in [
    ("raw training", raw_training),
    ("raw evaluation", raw_evaluation),
    ("model training", model_training),
    ("model evaluation", model_evaluation),
]:
    frame["Build"] = parse_int(
        frame["Build"],
        f"{label}.Build",
    )
    frame["Test"] = parse_int(
        frame["Test"],
        f"{label}.Test",
    )
    frame["Verdict"] = parse_int(
        frame["Verdict"],
        f"{label}.Verdict",
    )

raw_order_column = "RawTrainingRowOrder"
raw_eval_order_column = "RawEvaluationRowOrder"
model_train_order_column = "ModelTrainingRowOrder"
model_eval_order_column = "ModelEvaluationRowOrder"

for column, frame, expected_rows, label in [
    (
        raw_order_column,
        raw_training,
        EXPECTED_RAW_TRAIN_ROWS,
        "raw training",
    ),
    (
        raw_eval_order_column,
        raw_evaluation,
        EXPECTED_RAW_EVAL_ROWS,
        "raw evaluation",
    ),
    (
        model_train_order_column,
        model_training,
        EXPECTED_MODEL_TRAIN_ROWS,
        "model training",
    ),
    (
        model_eval_order_column,
        model_evaluation,
        EXPECTED_MODEL_EVAL_ROWS,
        "model evaluation",
    ),
]:
    if column not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing {column}."
        )

    frame[column] = parse_int(
        frame[column],
        f"{label}.{column}",
    )

    frame.sort_values(
        column,
        kind="mergesort",
        inplace=True,
    )
    frame.reset_index(drop=True, inplace=True)

    expected_sequence = np.arange(
        1,
        expected_rows + 1,
        dtype=np.int64,
    )

    if not np.array_equal(
        frame[column].to_numpy(dtype=np.int64),
        expected_sequence,
    ):
        raise RuntimeError(
            f"{label} row-order sequence is not canonical."
        )

model_train_link[model_train_order_column] = parse_int(
    model_train_link[model_train_order_column],
    "model_train_link.ModelTrainingRowOrder",
)
model_train_link[raw_order_column] = parse_int(
    model_train_link[raw_order_column],
    "model_train_link.RawTrainingRowOrder",
)
model_eval_link[model_eval_order_column] = parse_int(
    model_eval_link[model_eval_order_column],
    "model_eval_link.ModelEvaluationRowOrder",
)
model_eval_link[raw_eval_order_column] = parse_int(
    model_eval_link[raw_eval_order_column],
    "model_eval_link.RawEvaluationRowOrder",
)

model_train_link = (
    model_train_link.sort_values(
        model_train_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)
model_eval_link = (
    model_eval_link.sort_values(
        model_eval_order_column,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

model_training_raw_indices = (
    model_train_link[raw_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)
model_evaluation_raw_indices = (
    model_eval_link[raw_eval_order_column]
    .to_numpy(dtype=np.int64)
    - 1
)

if (
    model_training_raw_indices.min() < 0
    or model_training_raw_indices.max() >= EXPECTED_RAW_TRAIN_ROWS
):
    raise RuntimeError(
        "Model/raw training indices are outside the frozen raw cohort."
    )

if (
    model_evaluation_raw_indices.min() < 0
    or model_evaluation_raw_indices.max() >= EXPECTED_RAW_EVAL_ROWS
):
    raise RuntimeError(
        "Model/raw evaluation indices are outside the frozen raw cohort."
    )

linked_train_build = raw_training.iloc[
    model_training_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_train_test = raw_training.iloc[
    model_training_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_train_verdict = raw_training.iloc[
    model_training_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

linked_eval_build = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Build"].to_numpy(dtype=np.int64)
linked_eval_test = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Test"].to_numpy(dtype=np.int64)
linked_eval_verdict = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Verdict"].to_numpy(dtype=np.int64)

if not np.array_equal(
    linked_train_build,
    model_training["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Build links differ.")
if not np.array_equal(
    linked_train_test,
    model_training["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training Test links differ.")
if not np.array_equal(
    linked_train_verdict,
    model_training["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw training verdict links differ.")
if not np.array_equal(
    linked_eval_build,
    model_evaluation["Build"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Build links differ.")
if not np.array_equal(
    linked_eval_test,
    model_evaluation["Test"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation Test links differ.")
if not np.array_equal(
    linked_eval_verdict,
    model_evaluation["Verdict"].to_numpy(dtype=np.int64),
):
    raise RuntimeError("Model/raw evaluation verdict links differ.")

condition_plan["ConditionOrder"] = parse_int(
    condition_plan["ConditionOrder"],
    "condition_plan.ConditionOrder",
)
condition_plan["NoisePercent"] = parse_int(
    condition_plan["NoisePercent"],
    "condition_plan.NoisePercent",
)
condition_plan["RepetitionSeed"] = parse_int(
    condition_plan["RepetitionSeed"],
    "condition_plan.RepetitionSeed",
)

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if len(condition_plan) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "The frozen condition plan does not contain 270 conditions."
    )

if not np.array_equal(
    condition_plan["ConditionOrder"].to_numpy(dtype=np.int64),
    np.arange(1, EXPECTED_CONDITIONS + 1, dtype=np.int64),
):
    raise RuntimeError(
        "The frozen condition-order sequence is not canonical."
    )

if sorted(condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError(
        "The frozen noise-level set differs."
    )

if sorted(condition_plan["RepetitionSeed"].unique().tolist()) != REPETITION_SEEDS:
    raise RuntimeError(
        "The frozen repetition-seed set differs."
    )

if condition_plan["ConditionID"].duplicated(keep=False).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate condition IDs."
    )

if condition_plan.duplicated(
    subset=["NoisePercent", "RepetitionSeed"],
    keep=False,
).any():
    raise RuntimeError(
        "The frozen condition plan contains duplicate coordinates."
    )

rng_metadata_rows = int(
    pq.ParquetFile(RNG_MANIFEST_PATH).metadata.num_rows
)

if rng_metadata_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG-manifest row count differs."
    )

# --------------------------------------------------------------------------------------------------
# 7. PREDICTOR ORDER, NUMERIC MATRICES, CHRONOLOGY, ENTITY MAP, AND EVALUATION META
# --------------------------------------------------------------------------------------------------

if "Predictor" not in predictor_contract.columns:
    raise RuntimeError(
        "Predictor contract is missing the Predictor column."
    )

predictor_columns = predictor_contract[
    "Predictor"
].astype(str).tolist()

if len(predictor_columns) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract does not contain 151 predictors."
    )

if len(set(predictor_columns)) != EXPECTED_PREDICTORS:
    raise RuntimeError(
        "Predictor contract contains duplicate predictors."
    )

missing_training_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_training.columns
]
missing_evaluation_predictors = [
    feature
    for feature in predictor_columns
    if feature not in model_evaluation.columns
]

if missing_training_predictors or missing_evaluation_predictors:
    raise RuntimeError(
        "Frozen model cohorts are missing contract predictors."
    )

if any(feature not in predictor_columns for feature in REC_FEATURES):
    raise RuntimeError(
        "The 19 REC features are not all present in the predictor contract."
    )

if set(VERDICT_DEPENDENT_REC).intersection(
    VERDICT_INDEPENDENT_REC
):
    raise RuntimeError(
        "Dependent and independent REC sets overlap."
    )

if set(VERDICT_DEPENDENT_REC + VERDICT_INDEPENDENT_REC) != set(
    REC_FEATURES
):
    raise RuntimeError(
        "Dependent and independent REC sets do not partition all 19 REC features."
    )

print("Converting the fixed predictor cohorts to one numeric matrix.")

training_numeric_frame = model_training[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

evaluation_numeric_frame = model_evaluation[
    predictor_columns
].apply(
    pd.to_numeric,
    errors="coerce",
)

training_base_numeric = training_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)
evaluation_base_numeric = evaluation_numeric_frame.to_numpy(
    dtype=np.float64,
    copy=True,
)

training_base_numeric[
    ~np.isfinite(training_base_numeric)
] = np.nan
evaluation_base_numeric[
    ~np.isfinite(evaluation_base_numeric)
] = np.nan

all_base_numeric = np.vstack([
    training_base_numeric,
    evaluation_base_numeric,
])

predictor_index = {
    feature: index
    for index, feature in enumerate(predictor_columns)
}

dependent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_DEPENDENT_REC
    ],
    dtype=np.int64,
)

independent_predictor_indices = np.array(
    [
        predictor_index[feature]
        for feature in VERDICT_INDEPENDENT_REC
    ],
    dtype=np.int64,
)

model_all = pd.concat(
    [
        model_training[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
        model_evaluation[
            [
                "Build",
                "Test",
            ]
            + REC_FEATURES
        ],
    ],
    ignore_index=True,
)

if model_all.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Model cohort contains duplicate Build-Test rows."
    )

model_key_index = pd.MultiIndex.from_frame(
    model_all[["Build", "Test"]]
)

anchor_offsets = anchor_offsets.copy()
anchor_offsets["Build"] = parse_int(
    anchor_offsets["Build"],
    "anchor_offsets.Build",
)
anchor_offsets["Test"] = parse_int(
    anchor_offsets["Test"],
    "anchor_offsets.Test",
)

if anchor_offsets.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Anchor offsets contain duplicate Build-Test rows."
    )

anchor_indexed = anchor_offsets.set_index(
    [
        "Build",
        "Test",
    ]
)

missing_anchor_keys = model_key_index.difference(
    anchor_indexed.index
)

if len(missing_anchor_keys) != 0:
    raise RuntimeError(
        "Anchor offsets do not cover the full model cohort."
    )

anchor_values_all = anchor_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_reconstructed["Build"] = parse_int(
    clean_reconstructed["Build"],
    "clean_reconstructed.Build",
)
clean_reconstructed["Test"] = parse_int(
    clean_reconstructed["Test"],
    "clean_reconstructed.Test",
)

clean_reconstructed_indexed = clean_reconstructed.set_index(
    [
        "Build",
        "Test",
    ]
)

clean_reconstructed_all = clean_reconstructed_indexed.loc[
    model_key_index,
    REC_FEATURES,
].to_numpy(dtype=np.float64)

clean_anchored_all = (
    clean_reconstructed_all
    + anchor_values_all
)

clean_original_rec_all = model_all[
    REC_FEATURES
].to_numpy(dtype=np.float64)

clean_anchor_mismatch_values = int(
    (~np.isclose(
        clean_anchored_all,
        clean_original_rec_all,
        rtol=0,
        atol=1e-12,
    )).sum()
)

if clean_anchor_mismatch_values != 0:
    raise RuntimeError(
        "Frozen clean REC reconstruction plus anchor no longer reproduces the model cohort."
    )

# Project 24 Step 1B/2B chronology schema is BuildOrder / Build.
required_chronology_columns = {
    "BuildOrder",
    "Build",
}
if not required_chronology_columns.issubset(chronology.columns):
    raise RuntimeError(
        "Frozen Project 24 chronology is missing BuildOrder/Build."
    )

chronology["Build"] = parse_int(
    chronology["Build"],
    "chronology.Build",
)
chronology["BuildOrder"] = parse_int(
    chronology["BuildOrder"],
    "chronology.BuildOrder",
)

if not np.array_equal(
    chronology.sort_values(
        "BuildOrder",
        kind="mergesort",
    )["BuildOrder"].to_numpy(dtype=np.int64),
    np.arange(1, len(chronology) + 1, dtype=np.int64),
):
    raise RuntimeError(
        "Frozen Project 24 BuildOrder is not exactly 1..N."
    )

build_order_map = (
    chronology.set_index("Build")[
        "BuildOrder"
    ]
    .astype(int)
    .to_dict()
)

ordered_builds = (
    chronology.sort_values(
        "BuildOrder",
        kind="mergesort",
    )["Build"]
    .astype(int)
    .tolist()
)

global_build_position = {
    int(build_id): position
    for position, build_id in enumerate(ordered_builds)
}

# Project 24 Step 2A/2B build-entity schema is Build / EntityId.
required_build_entity_columns = {
    "Build",
    "EntityId",
}
if not required_build_entity_columns.issubset(build_entity.columns):
    raise RuntimeError(
        "Frozen Project 24 build-entity map is missing Build/EntityId."
    )

build_entity["Build"] = parse_int(
    build_entity["Build"],
    "build_entity.Build",
)
build_entity["EntityId"] = parse_int(
    build_entity["EntityId"],
    "build_entity.EntityId",
)

changed_entities_by_build = (
    build_entity.groupby(
        "Build"
    )["EntityId"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

changed_entities_by_build = {
    int(build_id): set(
        int(entity_id)
        for entity_id in changed_entities_by_build.get(
            int(build_id),
            set(),
        )
    )
    for build_id in ordered_builds
}

entity_changed_builds = (
    build_entity.groupby(
        "EntityId"
    )["Build"]
    .apply(
        lambda values: set(
            values.astype(int).tolist()
        )
    )
    .to_dict()
)

entity_changed_builds = {
    int(entity_id): set(
        int(build_id)
        for build_id in build_ids
    )
    for entity_id, build_ids in entity_changed_builds.items()
}

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "Job" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Job."
        )
    if "Duration" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing Duration."
        )

    frame["Duration"] = pd.to_numeric(
        frame["Duration"],
        errors="coerce",
    )

    if not np.isfinite(
        frame["Duration"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{label} cohort contains non-finite durations."
        )

    if frame["Duration"].lt(0).any():
        raise RuntimeError(
            f"{label} cohort contains negative durations."
        )

clean_raw_training_verdict = raw_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_raw_evaluation_verdict = raw_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_training_verdict = model_training[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_verdict = model_evaluation[
    "Verdict"
].to_numpy(dtype=np.int16)
clean_model_evaluation_binary = (
    clean_model_evaluation_verdict != 0
).astype(np.int8)

if int((clean_model_training_verdict != 0).sum()) != EXPECTED_MODEL_TRAIN_FAILURES:
    raise RuntimeError(
        "Clean model-training failure count differs."
    )

if int(clean_model_evaluation_binary.sum()) != EXPECTED_MODEL_EVAL_FAILURES:
    raise RuntimeError(
        "Clean model-evaluation failure count differs."
    )

evaluation_duration = raw_evaluation.iloc[
    model_evaluation_raw_indices
]["Duration"].to_numpy(dtype=float)

if not np.isfinite(evaluation_duration).all():
    raise RuntimeError(
        "Model evaluation durations are non-finite."
    )

failing_evaluation_builds = sorted(
    model_evaluation.loc[
        clean_model_evaluation_binary == 1,
        "Build",
    ]
    .astype(int)
    .unique()
    .tolist()
)

if len(failing_evaluation_builds) != EXPECTED_FAILING_EVAL_BUILDS:
    raise RuntimeError(
        "Failing evaluation-build count differs."
    )

evaluation_meta_base = pd.DataFrame({
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Build": model_evaluation["Build"].to_numpy(dtype=np.int64),
    "Test": model_evaluation["Test"].to_numpy(dtype=np.int64),
    "CleanVerdict": clean_model_evaluation_verdict.astype(np.int64),
    "CleanFailure": clean_model_evaluation_binary.astype(np.int8),
    "Duration": evaluation_duration.astype(float),
})

if evaluation_meta_base.duplicated(
    subset=["Build", "Test"],
    keep=False,
).any():
    raise RuntimeError(
        "Evaluation metadata contains duplicate Build-Test rows."
    )

# --------------------------------------------------------------------------------------------------
# 7B. PRECOMPUTE THE ACCELERATED REC ENGINE
# --------------------------------------------------------------------------------------------------

print("Precomputing the vectorized verdict-dependent REC engine.")

for frame, label in [
    (raw_training, "raw training"),
    (raw_evaluation, "raw evaluation"),
]:
    if "InferredTestOrder" not in frame.columns:
        raise RuntimeError(
            f"{label} cohort is missing the Step-2B-frozen InferredTestOrder column."
        )

    frame["InferredTestOrder"] = parse_int(
        frame["InferredTestOrder"],
        f"{label}.InferredTestOrder",
    )

combined_history = pd.DataFrame({
    "CombinedRowIndex": np.arange(
        EXPECTED_RAW_TRAIN_ROWS + EXPECTED_RAW_EVAL_ROWS,
        dtype=np.int64,
    ),
    "Build": np.concatenate((
        raw_training["Build"].to_numpy(dtype=np.int64),
        raw_evaluation["Build"].to_numpy(dtype=np.int64),
    )),
    "Test": np.concatenate((
        raw_training["Test"].to_numpy(dtype=np.int64),
        raw_evaluation["Test"].to_numpy(dtype=np.int64),
    )),
    "InferredTestOrder": np.concatenate((
        raw_training["InferredTestOrder"].to_numpy(dtype=np.int64),
        raw_evaluation["InferredTestOrder"].to_numpy(dtype=np.int64),
    )),
})

if combined_history.duplicated(
    subset=["Test", "InferredTestOrder"],
    keep=False,
).any():
    raise RuntimeError(
        "Accelerated REC history contains duplicate frozen per-test order keys."
    )

# Project 24 must use the exact per-test execution order frozen by Step 2B.
# Project 24 has 23 frozen timestamp-tie groups; InferredTestOrder is the authoritative per-test order from Step 2B.
combined_history = (
    combined_history.sort_values(
        ["Test", "InferredTestOrder"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

accelerated_history_combined_indices = combined_history[
    "CombinedRowIndex"
].to_numpy(dtype=np.int64)
accelerated_history_builds = combined_history[
    "Build"
].to_numpy(dtype=np.int64)
accelerated_history_tests = combined_history[
    "Test"
].to_numpy(dtype=np.int64)

accelerated_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        accelerated_history_tests[1:]
        != accelerated_history_tests[:-1]
    ).astype(np.int64) + 1,
))
accelerated_group_ends = np.concatenate((
    accelerated_group_starts[1:],
    np.array([len(combined_history)], dtype=np.int64),
))
accelerated_group_count = len(accelerated_group_starts)

if accelerated_group_count != int(combined_history["Test"].nunique()):
    raise RuntimeError(
        "Accelerated REC test-group count differs."
    )

history_key_index = pd.MultiIndex.from_arrays([
    accelerated_history_builds,
    accelerated_history_tests,
])

if not history_key_index.is_unique:
    raise RuntimeError(
        "Accelerated REC history contains duplicate Build-Test keys."
    )

model_history_positions = history_key_index.get_indexer(
    model_key_index
)

if (model_history_positions < 0).any():
    raise RuntimeError(
        "Accelerated REC history does not cover every model row."
    )

model_group_indices = np.searchsorted(
    accelerated_group_starts,
    model_history_positions,
    side="right",
) - 1
model_local_positions = (
    model_history_positions
    - accelerated_group_starts[model_group_indices]
)

accelerated_requested_model_indices = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]
accelerated_requested_local_positions = [
    np.empty(0, dtype=np.int64)
    for _ in range(accelerated_group_count)
]

request_order = np.lexsort((
    model_local_positions,
    model_group_indices,
))
ordered_group_indices = model_group_indices[request_order]
request_group_starts = np.concatenate((
    np.array([0], dtype=np.int64),
    np.flatnonzero(
        ordered_group_indices[1:]
        != ordered_group_indices[:-1]
    ).astype(np.int64) + 1,
))
request_group_ends = np.concatenate((
    request_group_starts[1:],
    np.array([len(request_order)], dtype=np.int64),
))

for request_start, request_end in zip(
    request_group_starts,
    request_group_ends,
):
    selected = request_order[request_start:request_end]
    group_index = int(model_group_indices[selected[0]])
    accelerated_requested_model_indices[group_index] = selected.astype(
        np.int64,
        copy=False,
    )
    accelerated_requested_local_positions[group_index] = model_local_positions[
        selected
    ].astype(np.int64, copy=False)

accelerated_entity_values = np.sort(
    build_entity["EntityId"].unique().astype(np.int64)
)
accelerated_entity_count = len(accelerated_entity_values)
accelerated_entity_to_dense = {
    int(entity_id): dense_index
    for dense_index, entity_id in enumerate(accelerated_entity_values)
}

accelerated_build_values = np.asarray(
    ordered_builds,
    dtype=np.int64,
)
accelerated_build_to_dense = {
    int(build_id): dense_index
    for dense_index, build_id in enumerate(accelerated_build_values)
}
accelerated_build_entity_arrays = [
    np.empty(0, dtype=np.int32)
    for _ in accelerated_build_values
]

for build_id, entity_ids in build_entity.groupby(
    "Build",
    sort=False,
)["EntityId"]:
    build_dense = accelerated_build_to_dense[int(build_id)]
    accelerated_build_entity_arrays[build_dense] = np.asarray(
        sorted({
            accelerated_entity_to_dense[int(entity_id)]
            for entity_id in entity_ids
        }),
        dtype=np.int32,
    )

accelerated_history_build_dense_indices = np.asarray([
    accelerated_build_to_dense[int(build_id)]
    for build_id in accelerated_history_builds
], dtype=np.int32)

accelerated_dependent_feature_indices = np.asarray([
    REC_FEATURES.index(feature)
    for feature in VERDICT_DEPENDENT_REC
], dtype=np.int64)
accelerated_anchor_dependent_all = anchor_values_all[
    :, accelerated_dependent_feature_indices
]

# Exact clean-equivalence self-test before any full condition is allowed.
accelerated_clean_combined_verdict = np.concatenate((
    clean_raw_training_verdict,
    clean_raw_evaluation_verdict,
)).astype(np.int16, copy=False)
accelerated_clean_reconstructed = reconstruct_dependent_rec_fast(
    accelerated_clean_combined_verdict
)
accelerated_clean_anchored = (
    accelerated_clean_reconstructed
    + accelerated_anchor_dependent_all
)
accelerated_clean_original = clean_original_rec_all[
    :, accelerated_dependent_feature_indices
]
accelerated_clean_mismatch_values = int((
    ~np.isclose(
        accelerated_clean_anchored,
        accelerated_clean_original,
        rtol=0,
        atol=1e-12,
    )
).sum())

if accelerated_clean_mismatch_values != 0:
    raise RuntimeError(
        "Accelerated REC engine failed the exact clean-data equivalence test."
    )

print(
    "Accelerated REC engine clean-equivalence mismatches:",
    accelerated_clean_mismatch_values,
)
print(
    "Accelerated REC groups / model rows / entities:",
    accelerated_group_count,
    "/",
    EXPECTED_MODEL_ROWS,
    "/",
    accelerated_entity_count,
)

# --------------------------------------------------------------------------------------------------
# 7B. WORKER SHARD FREEZE AND MASTER-WRITE GUARD
# --------------------------------------------------------------------------------------------------

worker_condition_plan = (
    condition_plan.loc[
        condition_plan["RepetitionSeed"].isin(WORKER_REPETITION_SEEDS)
    ]
    .sort_values("ConditionOrder", kind="mergesort")
    .reset_index(drop=True)
)

if len(worker_condition_plan) != EXPECTED_WORKER_CONDITIONS:
    raise RuntimeError(
        "Worker condition shard does not contain the expected number of conditions.\n"
        f"Expected: {EXPECTED_WORKER_CONDITIONS}\n"
        f"Actual:   {len(worker_condition_plan)}"
    )

if sorted(worker_condition_plan["RepetitionSeed"].unique().tolist()) != WORKER_REPETITION_SEEDS:
    raise RuntimeError("Worker repetition-seed shard differs from its frozen assignment.")

if sorted(worker_condition_plan["NoisePercent"].unique().tolist()) != NOISE_LEVELS:
    raise RuntimeError("Worker noise-level set differs from the frozen 9-level protocol.")

MASTER_WRITE_GUARD_PATHS = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    STEP5A_VALIDATION_PATH,
    STEP5A_REPORT_PATH,
    MASTER_RUN_PROGRESS_PATH,
    STEP5A_STATUS_PATH,
    STEP5A_CHECKPOINT_PATH,
]
master_write_guard_before = {
    str(path): path_guard_state(path)
    for path in MASTER_WRITE_GUARD_PATHS
}

print("Worker shard frozen:", WORKER_REPETITION_SEEDS)
print("Worker conditions:", len(worker_condition_plan))
print("Master Step 5A write guard armed for", len(MASTER_WRITE_GUARD_PATHS), "paths.")

# --------------------------------------------------------------------------------------------------
# 8. CHECKPOINT SCAN AND ASSIGNED CONDITION RUNNER
# --------------------------------------------------------------------------------------------------

FULL_RAW_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
FULL_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
INCOMPLETE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

raw_training_hash_before = sha256_file(RAW_TRAINING_COHORT_PATH)
raw_evaluation_hash_before = sha256_file(RAW_EVALUATION_COHORT_PATH)
model_training_hash_before = sha256_file(MODEL_TRAINING_COHORT_PATH)
model_evaluation_hash_before = sha256_file(MODEL_EVALUATION_COHORT_PATH)

expected_condition_files = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}

def validate_completed_condition(condition_dir, plan_row):
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != expected_condition_files:
        return None

    completion_path = condition_dir / "COMPLETE.json"
    summary_path = condition_dir / "condition_summary.json"

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception:
        return None

    if completion.get("Status") != CONDITION_STATUS:
        return None
    if summary.get("Status") != CONDITION_STATUS:
        return None
    if completion.get("ConditionKey") != condition_key:
        return None
    if summary.get("ConditionKey") != condition_key:
        return None
    if int(summary.get("NoisePercent", -1)) != int(plan_row.NoisePercent):
        return None
    if int(summary.get("RepetitionSeed", -1)) != int(plan_row.RepetitionSeed):
        return None
    if str(completion.get("ConditionSummaryPath")) != str(summary_path):
        return None
    if str(completion.get("ConditionSummarySHA256")) != sha256_file(summary_path):
        return None

    output_manifest = summary.get("OutputManifest", [])
    if not isinstance(output_manifest, list) or len(output_manifest) != 6:
        return None

    for item in output_manifest:
        path = Path(item.get("Path", ""))
        if path.parent != condition_dir:
            return None
        if not path.is_file():
            return None
        if int(path.stat().st_size) != int(item.get("Bytes", -1)):
            return None
        if sha256_file(path) != str(item.get("SHA256", "")):
            return None

    expected_counts = {
        "MLFits": EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows": EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows": EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows": EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows": EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(summary.get(key, -1)) != expected:
            return None

    if int(summary.get("IndependentRECChanges", -1)) != 0:
        return None

    fingerprints = summary.get("BaselineFingerprints", {})
    if sorted(fingerprints.keys()) != ["QTF-Avg", "Random"]:
        return None

    condition_manifest = directory_manifest(condition_dir)
    if len(condition_manifest) != EXPECTED_FILES_PER_CONDITION:
        return None

    return {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": int(plan_row.ConditionOrder),
        "ConditionKey": condition_key,
        "NoisePercent": int(plan_row.NoisePercent),
        "RepetitionSeed": int(plan_row.RepetitionSeed),
        "ConditionDirectory": str(condition_dir),
        "Status": CONDITION_STATUS,
        "Files": int(len(condition_manifest)),
        "ConditionBytes": int(condition_manifest["Bytes"].sum()),
        "ConditionRootSHA256": directory_root_hash(condition_manifest),
        "RankingRows": int(summary["RankingRows"]),
        "BuildMetricRows": int(summary["BuildMetricRows"]),
        "ProjectRunRows": int(summary["ProjectRunRows"]),
        "ModelFitRows": int(summary["MLFits"]),
        "TrainingMedianRows": int(summary["TrainingMedianRows"]),
        "ConditionSeconds": float(summary["ConditionSeconds"]),
        "RandomScoreSHA256": str(fingerprints["Random"]["ScoreSHA256"]),
        "RandomRankSHA256": str(fingerprints["Random"]["RankSHA256"]),
        "QTFAvgScoreSHA256": str(fingerprints["QTF-Avg"]["ScoreSHA256"]),
        "QTFAvgRankSHA256": str(fingerprints["QTF-Avg"]["RankSHA256"]),
    }

def quarantine_incomplete_condition(condition_dir):
    condition_dir = Path(condition_dir)

    if not condition_dir.exists():
        return None

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = INCOMPLETE_BACKUP_ROOT / f"{condition_dir.name}__{timestamp}"
    shutil.move(str(condition_dir), str(destination))
    return destination

print("\nScanning existing condition checkpoints...")

valid_existing = {}
invalid_existing = []

for plan_row in worker_condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is not None:
        valid_existing[condition_key] = validated
    elif condition_dir.exists():
        invalid_existing.append(condition_key)

print("Valid completed conditions:", len(valid_existing))
print("Incomplete/invalid condition directories:", len(invalid_existing))
print("Pending assigned conditions:", EXPECTED_WORKER_CONDITIONS - len(valid_existing))

for condition_key in invalid_existing:
    backup = quarantine_incomplete_condition(
        FULL_RAW_RESULT_ROOT / condition_key
    )
    print("Preserved incomplete condition in:", backup)

# The seed-1 worker owns the shared accelerated-equivalence audit.
# Other workers may start simultaneously but wait here until that exact two-row PASS audit exists.
equivalence_records_by_key = {}

if WORKER_IS_SMOKE_OWNER:
    for equivalence_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_row = condition_plan.loc[
            condition_plan["ConditionID"].eq(equivalence_key)
        ]
        if len(equivalence_row) != 1:
            raise RuntimeError(
                f"Could not uniquely resolve smoke-equivalence condition {equivalence_key}."
            )
        equivalence_plan_row = next(equivalence_row.itertuples(index=False))
        equivalence_dir = FULL_RAW_RESULT_ROOT / equivalence_key
        if validate_completed_condition(equivalence_dir, equivalence_plan_row) is not None:
            equivalence_record = compare_full_condition_to_smoke(
                equivalence_key,
                equivalence_dir,
            )
            if not equivalence_record["Pass"]:
                raise RuntimeError(
                    f"Existing accelerated condition {equivalence_key} differs from Step 4B."
                )
            equivalence_records_by_key[equivalence_key] = equivalence_record

    smoke_first_plan = worker_condition_plan.loc[
        worker_condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
    ].copy()
    smoke_first_plan["__SmokeOrder"] = smoke_first_plan["ConditionID"].map({
        key: index
        for index, key in enumerate(SMOKE_EQUIVALENCE_KEYS)
    })
    smoke_first_plan = smoke_first_plan.sort_values(
        "__SmokeOrder",
        kind="mergesort",
    ).drop(columns="__SmokeOrder")
    remaining_plan = worker_condition_plan.loc[
        ~worker_condition_plan["ConditionID"].isin(SMOKE_EQUIVALENCE_KEYS)
    ].sort_values("ConditionOrder", kind="mergesort")
    execution_plan = pd.concat(
        [smoke_first_plan, remaining_plan],
        ignore_index=True,
    )
else:
    shared_equivalence = wait_for_shared_equivalence()
    equivalence_records_by_key = {
        str(row.ConditionKey): {"Pass": True}
        for row in shared_equivalence.itertuples(index=False)
    }
    execution_plan = worker_condition_plan.copy()

if len(execution_plan) != EXPECTED_WORKER_CONDITIONS:
    raise RuntimeError(
        "Worker accelerated execution plan does not contain its exact assigned conditions."
    )

if WORKER_IS_SMOKE_OWNER and len(equivalence_records_by_key) == len(SMOKE_EQUIVALENCE_KEYS):
    atomic_csv(
        ACCELERATED_EQUIVALENCE_PATH,
        pd.DataFrame(equivalence_records_by_key.values()).sort_values(
            "ConditionKey",
            kind="mergesort",
        ),
    )

rng_parquet_file = pq.ParquetFile(
    RNG_MANIFEST_PATH
)

if int(rng_parquet_file.metadata.num_rows) != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs before worker execution."
    )

if int(rng_parquet_file.metadata.num_row_groups) != 30:
    raise RuntimeError(
        "Frozen RNG manifest must contain exactly 30 one-seed row groups."
    )

full_execution_started = time.perf_counter()
completed_this_run = 0
skipped_valid = 0
current_rng_seed = None
flip_uniform = None
sampled_failure_subtype = None
random_scores = None

for worker_position, plan_row in enumerate(execution_plan.itertuples(index=False), start=1):
    condition_order = int(plan_row.ConditionOrder)
    condition_key = str(plan_row.ConditionID)
    noise_percent = int(plan_row.NoisePercent)
    repetition_seed = int(plan_row.RepetitionSeed)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key

    already_valid = validate_completed_condition(condition_dir, plan_row)
    if already_valid is not None:
        skipped_valid += 1
        print(
            f"[worker {worker_position}/{EXPECTED_WORKER_CONDITIONS} | global {condition_order}/{EXPECTED_CONDITIONS}] "
            f"Skipping validated checkpoint {condition_key}"
        )
        continue

    if (
        condition_key not in SMOKE_EQUIVALENCE_KEYS
        and set(equivalence_records_by_key) != set(SMOKE_EQUIVALENCE_KEYS)
    ):
        raise RuntimeError(
            "The accelerated engine must pass both frozen smoke-output equivalence checks "
            "before any other full condition is executed."
        )

    if repetition_seed != current_rng_seed:
        print(f"\nLoading deterministic RNG stream for seed {repetition_seed}.")

        rng_seed_frame = rng_parquet_file.read_row_group(
            repetition_seed - 1,
            columns=[
                "RepetitionSeed",
                "RawTrainingRowOrder",
                "FlipUniform",
                "SampledFailureSubtype",
            ],
        ).to_pandas()

        if (
            rng_seed_frame["RepetitionSeed"].nunique() != 1
            or int(rng_seed_frame["RepetitionSeed"].iloc[0]) != repetition_seed
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row-group seed identity differs."
            )
        rng_seed_frame[raw_order_column] = parse_int(
            rng_seed_frame[raw_order_column],
            f"rng_seed_{repetition_seed}.RawTrainingRowOrder",
        )
        rng_seed_frame = (
            rng_seed_frame.sort_values(
                raw_order_column,
                kind="mergesort",
            )
            .reset_index(drop=True)
        )

        if len(rng_seed_frame) != EXPECTED_RAW_TRAIN_ROWS:
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG stream row count differs."
            )

        if not np.array_equal(
            rng_seed_frame[raw_order_column].to_numpy(dtype=np.int64),
            np.arange(1, EXPECTED_RAW_TRAIN_ROWS + 1, dtype=np.int64),
        ):
            raise RuntimeError(
                f"Seed {repetition_seed}: RNG row order differs."
            )

        flip_uniform = rng_seed_frame["FlipUniform"].to_numpy(dtype=np.float64)
        sampled_failure_subtype = rng_seed_frame[
            "SampledFailureSubtype"
        ].to_numpy(dtype=np.int16)

        if not np.isfinite(flip_uniform).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: non-finite flip uniforms."
            )
        if ((flip_uniform < 0) | (flip_uniform >= 1)).any():
            raise RuntimeError(
                f"Seed {repetition_seed}: flip uniforms outside [0,1)."
            )

        # Exact Step 4B Random-baseline assignment:
        # draw per build after sorting that build's rows by Test, then place scores
        # back into the fixed evaluation-row positions.
        random_scores = np.empty(EXPECTED_MODEL_EVAL_ROWS, dtype=np.float64)
        eval_build_array = evaluation_meta_base["Build"].to_numpy(dtype=np.int64)
        eval_test_array = evaluation_meta_base["Test"].to_numpy(dtype=np.int64)

        for build_id in sorted(evaluation_meta_base["Build"].unique()):
            build_indices = np.flatnonzero(
                eval_build_array == int(build_id)
            )
            ordered_indices = build_indices[
                np.argsort(
                    eval_test_array[build_indices],
                    kind="mergesort",
                )
            ]
            random_scores[ordered_indices] = np.random.default_rng(
                deterministic_random_build_seed(
                    repetition_seed,
                    int(build_id),
                )
            ).random(len(ordered_indices))

        if not np.isfinite(random_scores).all():
            raise RuntimeError(
                f"Seed {repetition_seed}: Random baseline scores are non-finite."
            )

        current_rng_seed = repetition_seed
        del rng_seed_frame
        gc.collect()

    condition_started = time.perf_counter()

    print("\n" + "-" * 110)
    print(
        f"[worker {worker_position}/{EXPECTED_WORKER_CONDITIONS} | global {condition_order}/{EXPECTED_CONDITIONS}] Running {condition_key}"
    )
    print("-" * 110)

    condition_dir.mkdir(parents=True, exist_ok=True)

    ranking_path = condition_dir / "rankings.csv.gz"
    build_metrics_path = condition_dir / "build_metrics.csv"
    project_runs_path = condition_dir / "project_runs.csv"
    model_fits_path = condition_dir / "model_fits.csv"
    training_medians_path = condition_dir / "training_medians.csv"
    condition_audit_path = condition_dir / "condition_audit.csv"
    condition_summary_path = condition_dir / "condition_summary.json"
    completion_marker_path = condition_dir / "COMPLETE.json"

    flip_mask = flip_uniform < (noise_percent / 100.0)
    noisy_raw_training_verdict = clean_raw_training_verdict.copy()

    pass_to_failure_mask = flip_mask & (clean_raw_training_verdict == 0)
    failure_to_pass_mask = flip_mask & (clean_raw_training_verdict != 0)

    noisy_raw_training_verdict[pass_to_failure_mask] = (
        sampled_failure_subtype[pass_to_failure_mask]
    )
    noisy_raw_training_verdict[failure_to_pass_mask] = 0

    noisy_model_training_verdict = noisy_raw_training_verdict[
        model_training_raw_indices
    ]

    actual_flip_mask_sha256 = sha256_array(
        flip_mask.astype(np.uint8),
        "u1",
    )
    actual_noisy_raw_sha256 = sha256_array(
        noisy_raw_training_verdict,
        "<i2",
    )
    actual_noisy_model_sha256 = sha256_array(
        noisy_model_training_verdict,
        "<i2",
    )

    expected_flip_mask_sha256 = str(plan_row.FlipMaskSHA256)
    expected_noisy_raw_sha256 = str(plan_row.NoisyRawVerdictSHA256)
    expected_noisy_model_sha256 = str(plan_row.NoisyModelVerdictSHA256)

    if actual_flip_mask_sha256 != expected_flip_mask_sha256:
        raise RuntimeError(
            f"{condition_key}: flip-mask SHA-256 differs from Step 3A."
        )
    if actual_noisy_raw_sha256 != expected_noisy_raw_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy raw-verdict SHA-256 differs from Step 3A."
        )
    if actual_noisy_model_sha256 != expected_noisy_model_sha256:
        raise RuntimeError(
            f"{condition_key}: noisy model-verdict SHA-256 differs from Step 3A."
        )

    number_flipped = int(flip_mask.sum())
    pass_to_failure = int(pass_to_failure_mask.sum())
    failure_to_pass = int(failure_to_pass_mask.sum())
    model_label_changes = int(
        (noisy_model_training_verdict != clean_model_training_verdict).sum()
    )
    noisy_model_training_binary = (
        noisy_model_training_verdict != 0
    ).astype(np.int8)
    noisy_model_training_failures = int(noisy_model_training_binary.sum())

    if number_flipped != int(plan_row.NumberFlipped):
        raise RuntimeError(
            f"{condition_key}: NumberFlipped differs from Step 3A."
        )
    if pass_to_failure != int(plan_row.PassToFailure):
        raise RuntimeError(
            f"{condition_key}: PassToFailure differs from Step 3A."
        )
    if failure_to_pass != int(plan_row.FailureToPass):
        raise RuntimeError(
            f"{condition_key}: FailureToPass differs from Step 3A."
        )
    if model_label_changes != int(plan_row.ModelLabelChanges):
        raise RuntimeError(
            f"{condition_key}: ModelLabelChanges differs from Step 3A."
        )
    if noisy_model_training_failures != int(plan_row.NoisyModelFailures):
        raise RuntimeError(
            f"{condition_key}: NoisyModelFailures differs from Step 3A."
        )

    rec_started = time.perf_counter()

    if noise_percent == 0:
        # The 0% REC matrix is already frozen and exact. Reusing it avoids
        # thirty identical full-history reconstructions.
        condition_numeric_all = all_base_numeric.copy()
        dependent_rec_changes = 0
        independent_rec_changes = 0
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS
    else:
        condition_combined_verdict = np.concatenate((
            noisy_raw_training_verdict,
            clean_raw_evaluation_verdict,
        )).astype(np.int16, copy=False)

        reconstructed_dependent = reconstruct_dependent_rec_fast(
            condition_combined_verdict
        )
        anchored_dependent = (
            reconstructed_dependent
            + accelerated_anchor_dependent_all
        )

        condition_numeric_all = all_base_numeric.copy()
        condition_numeric_all[:, dependent_predictor_indices] = (
            anchored_dependent
        )

        dependent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, dependent_predictor_indices],
                all_base_numeric[:, dependent_predictor_indices],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        independent_rec_changes = int((
            ~np.isclose(
                condition_numeric_all[:, independent_predictor_indices],
                all_base_numeric[:, independent_predictor_indices],
                rtol=0,
                atol=0,
                equal_nan=True,
            )
        ).sum())
        independent_reconstruction_mismatches = 0
        reconstructed_row_count = EXPECTED_MODEL_ROWS

        if independent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: preserved independent REC predictors changed."
            )

    rec_seconds = time.perf_counter() - rec_started

    condition_training_numeric = condition_numeric_all[
        :EXPECTED_MODEL_TRAIN_ROWS
    ].copy()
    condition_evaluation_numeric = condition_numeric_all[
        EXPECTED_MODEL_TRAIN_ROWS:
    ].copy()

    if noise_percent == 0:
        zero_rec_mismatches = int((
            ~np.isclose(
                condition_numeric_all[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                all_base_numeric[:, [
                    predictor_index[feature]
                    for feature in REC_FEATURES
                ]],
                rtol=0,
                atol=1e-12,
                equal_nan=True,
            )
        ).sum())
        if zero_rec_mismatches != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition did not reproduce clean REC."
            )
        if number_flipped != 0 or model_label_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed labels."
            )
        if dependent_rec_changes != 0:
            raise RuntimeError(
                f"{condition_key}: 0% condition changed dependent REC."
            )
    else:
        if number_flipped <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no raw labels."
            )
        if model_label_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no model labels."
            )
        if dependent_rec_changes <= 0:
            raise RuntimeError(
                f"{condition_key}: positive noise changed no dependent REC."
            )

    medians = np.nanmedian(condition_training_numeric, axis=0)
    nonfinite_median_indices = np.flatnonzero(~np.isfinite(medians))
    if len(nonfinite_median_indices) != 0:
        bad_features = [predictor_columns[index] for index in nonfinite_median_indices]
        raise RuntimeError(
            f"{condition_key}: non-finite training medians for {bad_features}."
        )

    training_missing_mask = ~np.isfinite(condition_training_numeric)
    evaluation_missing_mask = ~np.isfinite(condition_evaluation_numeric)

    if training_missing_mask.any():
        row_indices, column_indices = np.where(training_missing_mask)
        condition_training_numeric[row_indices, column_indices] = medians[
            column_indices
        ]
    if evaluation_missing_mask.any():
        row_indices, column_indices = np.where(evaluation_missing_mask)
        condition_evaluation_numeric[row_indices, column_indices] = medians[
            column_indices
        ]

    if not np.isfinite(condition_training_numeric).all():
        raise RuntimeError(
            f"{condition_key}: training matrix remains non-finite."
        )
    if not np.isfinite(condition_evaluation_numeric).all():
        raise RuntimeError(
            f"{condition_key}: evaluation matrix remains non-finite."
        )

    training_medians = pd.DataFrame({
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "PredictorOrder": np.arange(
            1,
            EXPECTED_PREDICTORS + 1,
            dtype=np.int64,
        ),
        "Predictor": predictor_columns,
        "TrainingMedian": medians.astype(float),
    })

    evaluation_meta = evaluation_meta_base.copy()
    evaluation_meta["ConditionKey"] = condition_key
    evaluation_meta["NoisePercent"] = noise_percent
    evaluation_meta["RepetitionSeed"] = repetition_seed

    technique_scores = {}
    model_fit_records = []
    models = create_models(repetition_seed)

    for technique in ML_TECHNIQUES:
        print(f"  Fitting: {technique}")
        model = models[technique]
        fit_started = time.perf_counter()

        try:
            model.fit(
                condition_training_numeric,
                noisy_model_training_binary,
            )
            fit_seconds = time.perf_counter() - fit_started
            technique_scores[technique] = positive_probability(
                model,
                condition_evaluation_numeric,
            )
            fit_status = "PASS_MODEL_FIT"
            fit_error = ""
        except Exception as error:
            fit_seconds = time.perf_counter() - fit_started
            fit_status = "FAIL_MODEL_FIT"
            fit_error = repr(error)
            model_fit_records.append({
                "ProjectNumber": PROJECT_NUMBER,
                "Project": PROJECT_NAME,
                "ProjectSlug": PROJECT_SLUG,
                "ConditionKey": condition_key,
                "NoisePercent": noise_percent,
                "RepetitionSeed": repetition_seed,
                "Technique": technique,
                "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
                "TrainingFailures": noisy_model_training_failures,
                "Predictors": EXPECTED_PREDICTORS,
                "FitSeconds": float(fit_seconds),
                "ClassesJSON": "[]",
                "Status": fit_status,
                "Error": fit_error,
            })
            raise RuntimeError(
                f"{condition_key}: {technique} fitting failed: {error!r}"
            ) from error

        model_fit_records.append({
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "ConditionKey": condition_key,
            "NoisePercent": noise_percent,
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "TrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
            "TrainingFailures": noisy_model_training_failures,
            "Predictors": EXPECTED_PREDICTORS,
            "FitSeconds": float(fit_seconds),
            "ClassesJSON": json.dumps([
                int(value)
                for value in np.asarray(model.classes_).tolist()
            ]),
            "Status": fit_status,
            "Error": fit_error,
        })
        del model
        gc.collect()

    model_fits = pd.DataFrame(model_fit_records)
    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: model-fit row count differs."
        )
    if not model_fits["Status"].eq("PASS_MODEL_FIT").all():
        raise RuntimeError(
            f"{condition_key}: one or more model fits failed."
        )

    condition_eval_last_failure_age = condition_evaluation_numeric[
        :, predictor_index["REC_LastFailureAge"]
    ].astype(float)
    condition_eval_qtf = condition_evaluation_numeric[
        :, predictor_index["REC_TotalAvgExeTime"]
    ].astype(float)

    technique_scores["Random"] = random_scores.copy()
    technique_scores["LatestFail"] = -condition_eval_last_failure_age
    technique_scores["QTF-Avg"] = condition_eval_qtf

    ranking_frames = []
    for technique in ALL_TECHNIQUES:
        ranking_frames.append(
            make_ranking(
                evaluation_meta=evaluation_meta,
                technique=technique,
                scores=technique_scores[technique],
                ascending_score=(technique == "QTF-Avg"),
            )
        )

    rankings = pd.concat(ranking_frames, ignore_index=True)
    if len(rankings) != EXPECTED_RANKING_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: ranking row count differs."
        )
    if sorted(rankings["Technique"].unique().tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: ranking technique set differs."
        )
    if not rankings.groupby("Technique").size().eq(
        EXPECTED_MODEL_EVAL_ROWS
    ).all():
        raise RuntimeError(
            f"{condition_key}: ranking rows per technique differ."
        )
    if rankings.duplicated(
        subset=["Technique", "Build", "Test"],
        keep=False,
    ).any():
        raise RuntimeError(
            f"{condition_key}: duplicate ranking rows found."
        )

    build_metrics, project_runs = calculate_condition_metrics(rankings)
    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: build-metric row count differs."
        )
    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        raise RuntimeError(
            f"{condition_key}: project-run row count differs."
        )
    if sorted(project_runs["Technique"].tolist()) != sorted(ALL_TECHNIQUES):
        raise RuntimeError(
            f"{condition_key}: project-run technique set differs."
        )

    build_metric_values = build_metrics[["APFDc", "APFD"]].to_numpy(dtype=float)
    if not np.isfinite(build_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: build metrics contain non-finite values."
        )
    if ((build_metric_values < 0) | (build_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: build metrics fall outside [0,1]."
        )

    project_metric_values = project_runs[[
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]].to_numpy(dtype=float)
    if not np.isfinite(project_metric_values).all():
        raise RuntimeError(
            f"{condition_key}: project metrics contain non-finite values."
        )
    if ((project_metric_values < 0) | (project_metric_values > 1)).any():
        raise RuntimeError(
            f"{condition_key}: project metrics fall outside [0,1]."
        )

    condition_seconds = time.perf_counter() - condition_started

    condition_audit = pd.DataFrame([{
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "RawTrainingRows": EXPECTED_RAW_TRAIN_ROWS,
        "NumberFlipped": number_flipped,
        "ExpectedNumberFlipped": int(plan_row.NumberFlipped),
        "RealisedNoisePercent": float(
            100.0 * number_flipped / EXPECTED_RAW_TRAIN_ROWS
        ),
        "PassToFailure": pass_to_failure,
        "FailureToPass": failure_to_pass,
        "ModelTrainingRows": EXPECTED_MODEL_TRAIN_ROWS,
        "ModelLabelChanges": model_label_changes,
        "ExpectedModelLabelChanges": int(plan_row.ModelLabelChanges),
        "TrainingFailures": noisy_model_training_failures,
        "ExpectedTrainingFailures": int(plan_row.NoisyModelFailures),
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "IndependentReconstructionMismatches": independent_reconstruction_mismatches,
        "ReconstructedRows": int(reconstructed_row_count),
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ExpectedFlipMaskSHA256": expected_flip_mask_sha256,
        "ActualFlipMaskSHA256": actual_flip_mask_sha256,
        "ExpectedNoisyRawVerdictSHA256": expected_noisy_raw_sha256,
        "ActualNoisyRawVerdictSHA256": actual_noisy_raw_sha256,
        "ExpectedNoisyModelVerdictSHA256": expected_noisy_model_sha256,
        "ActualNoisyModelVerdictSHA256": actual_noisy_model_sha256,
        "RECSeconds": float(rec_seconds),
        "ConditionSeconds": float(condition_seconds),
        "Status": CONDITION_STATUS,
    }])

    baseline_fingerprints = {}
    for technique in ["Random", "QTF-Avg"]:
        baseline_rows = (
            rankings.loc[
                rankings["Technique"].eq(technique),
                ["Build", "Test", "Score", "Rank"],
            ]
            .sort_values(["Build", "Test"], kind="mergesort")
            .reset_index(drop=True)
        )
        baseline_fingerprints[technique] = {
            "Rows": int(len(baseline_rows)),
            "KeySHA256": hashlib.sha256(
                np.ascontiguousarray(
                    baseline_rows[["Build", "Test"]].to_numpy(dtype=np.int64)
                ).tobytes(order="C")
            ).hexdigest(),
            "ScoreSHA256": sha256_array(
                baseline_rows["Score"].to_numpy(dtype=np.float64),
                "<f8",
            ),
            "RankSHA256": sha256_array(
                baseline_rows["Rank"].to_numpy(dtype=np.int64),
                "<i8",
            ),
        }

    atomic_csv(ranking_path, rankings, compression="gzip")
    atomic_csv(build_metrics_path, build_metrics)
    atomic_csv(project_runs_path, project_runs)
    atomic_csv(model_fits_path, model_fits)
    atomic_csv(training_medians_path, training_medians)
    atomic_csv(condition_audit_path, condition_audit)

    if condition_key in SMOKE_EQUIVALENCE_KEYS:
        equivalence_record = compare_full_condition_to_smoke(
            condition_key,
            condition_dir,
        )
        if not equivalence_record["Pass"]:
            print("\nAccelerated-engine equivalence failure:")
            display(pd.DataFrame([equivalence_record]))
            raise RuntimeError(
                f"{condition_key}: accelerated outputs differ from the frozen Step 4B outputs."
            )
        equivalence_records_by_key[condition_key] = equivalence_record
        if WORKER_IS_SMOKE_OWNER:
            atomic_csv(
                ACCELERATED_EQUIVALENCE_PATH,
                pd.DataFrame(equivalence_records_by_key.values()).sort_values(
                    "ConditionKey",
                    kind="mergesort",
                ),
            )
        print(
            "  Frozen smoke-output equivalence: PASS |",
            condition_key,
        )

    condition_output_paths = [
        ranking_path,
        build_metrics_path,
        project_runs_path,
        model_fits_path,
        training_medians_path,
        condition_audit_path,
    ]
    condition_output_manifest = [
        {
            "Path": str(path),
            "Bytes": int(path.stat().st_size),
            "SHA256": sha256_file(path),
        }
        for path in condition_output_paths
    ]

    condition_summary = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "NoisePercent": noise_percent,
        "RepetitionSeed": repetition_seed,
        "Status": CONDITION_STATUS,
        "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
        "NumberFlipped": number_flipped,
        "ModelLabelChanges": model_label_changes,
        "DependentRECChanges": dependent_rec_changes,
        "IndependentRECChanges": independent_rec_changes,
        "TrainingFailures": noisy_model_training_failures,
        "Predictors": EXPECTED_PREDICTORS,
        "MLFits": int(len(model_fits)),
        "RankingRows": int(len(rankings)),
        "BuildMetricRows": int(len(build_metrics)),
        "ProjectRunRows": int(len(project_runs)),
        "TrainingMedianRows": int(len(training_medians)),
        "ConditionSeconds": float(condition_seconds),
        "BaselineFingerprints": baseline_fingerprints,
        "OutputManifest": condition_output_manifest,
    }
    atomic_json(condition_summary_path, condition_summary)

    completion_marker = {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "ConditionOrder": condition_order,
        "ConditionKey": condition_key,
        "Status": CONDITION_STATUS,
        "ConditionSummaryPath": str(condition_summary_path),
        "ConditionSummarySHA256": sha256_file(condition_summary_path),
        "CompletedAtUTC": datetime.now(timezone.utc).isoformat(),
    }
    atomic_json(completion_marker_path, completion_marker)

    validated_after_write = validate_completed_condition(
        condition_dir,
        plan_row,
    )
    if validated_after_write is None:
        raise RuntimeError(
            f"{condition_key}: completed condition did not pass readback validation."
        )

    completed_this_run += 1
    completed_total = skipped_valid + completed_this_run

    atomic_json(
        RUN_PROGRESS_PATH,
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": f"PROJECT_24_STEP5A_WORKER_{WORKER_TAG}_IN_PROGRESS",
            "UpdatedAtUTC": datetime.now(timezone.utc).isoformat(),
            "CompletedConditions": completed_total,
            "ExpectedConditions": EXPECTED_WORKER_CONDITIONS,
            "LastCompletedCondition": condition_key,
            "LastCompletedConditionOrder": condition_order,
            "ResumeSafe": True,
            "RegistryModified": False,
            "PriorProjectConditionOutputsAccessed": False,
        },
    )

    print(
        f"  Completed: {condition_key}\n"
        f"  Raw flips: {number_flipped} | "
        f"model-label changes: {model_label_changes} | "
        f"dependent REC changes: {dependent_rec_changes}\n"
        f"  Training failures: {noisy_model_training_failures} | "
        f"condition seconds: {condition_seconds:.2f}"
    )

    del condition_numeric_all
    if noise_percent > 0:
        del condition_combined_verdict
        del reconstructed_dependent
        del anchored_dependent
    del condition_training_numeric
    del condition_evaluation_numeric
    del rankings
    del ranking_frames
    del technique_scores
    del models
    gc.collect()

# --------------------------------------------------------------------------------------------------
# 9. FINAL ASSIGNED-SHARD REVALIDATION
# --------------------------------------------------------------------------------------------------

print(f"\nValidating all {EXPECTED_WORKER_CONDITIONS} assigned conditions for {WORKER_TAG}.")

inventory_records = []
condition_audits = []
baseline_records = []
worker_raw_manifest_records = []
model_fit_rows = 0
build_metric_rows = 0
project_run_rows = 0

for plan_row in worker_condition_plan.itertuples(index=False):
    condition_key = str(plan_row.ConditionID)
    condition_dir = FULL_RAW_RESULT_ROOT / condition_key
    validated = validate_completed_condition(condition_dir, plan_row)

    if validated is None:
        raise RuntimeError(
            f"Worker final validation failed for {condition_key}."
        )

    inventory_records.append(validated)
    condition_audits.append(pd.read_csv(condition_dir / "condition_audit.csv"))
    model_fit_rows += len(pd.read_csv(condition_dir / "model_fits.csv"))
    build_metric_rows += len(pd.read_csv(condition_dir / "build_metrics.csv"))
    project_run_rows += len(pd.read_csv(condition_dir / "project_runs.csv"))

    summary = load_json(condition_dir / "condition_summary.json")
    for technique in ["Random", "QTF-Avg"]:
        fingerprint = summary["BaselineFingerprints"][technique]
        baseline_records.append({
            "ConditionKey": condition_key,
            "NoisePercent": int(plan_row.NoisePercent),
            "RepetitionSeed": int(plan_row.RepetitionSeed),
            "Technique": technique,
            "Rows": int(fingerprint["Rows"]),
            "KeySHA256": str(fingerprint["KeySHA256"]),
            "ScoreSHA256": str(fingerprint["ScoreSHA256"]),
            "RankSHA256": str(fingerprint["RankSHA256"]),
        })

    condition_manifest = directory_manifest(condition_dir)
    for item in condition_manifest.itertuples(index=False):
        worker_raw_manifest_records.append({
            "RelativePath": f"{condition_key}/{item.RelativePath}",
            "Bytes": int(item.Bytes),
            "SHA256": str(item.SHA256),
        })

worker_inventory = pd.DataFrame(inventory_records).sort_values(
    "ConditionOrder",
    kind="mergesort",
).reset_index(drop=True)
combined_condition_audit = pd.concat(condition_audits, ignore_index=True)
baseline_fingerprints = pd.DataFrame(baseline_records)
worker_raw_manifest = pd.DataFrame(
    worker_raw_manifest_records,
    columns=DIRECTORY_MANIFEST_COLUMNS,
).sort_values("RelativePath", kind="mergesort").reset_index(drop=True)
worker_raw_root_sha256 = directory_root_hash(worker_raw_manifest)
worker_raw_files = int(len(worker_raw_manifest))
worker_raw_bytes = int(worker_raw_manifest["Bytes"].sum())

baseline_invariance_records = []
for repetition_seed in WORKER_REPETITION_SEEDS:
    for technique in ["Random", "QTF-Avg"]:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints["RepetitionSeed"].eq(repetition_seed)
            & baseline_fingerprints["Technique"].eq(technique)
        ]
        key_variants = int(rows["KeySHA256"].nunique())
        score_variants = int(rows["ScoreSHA256"].nunique())
        rank_variants = int(rows["RankSHA256"].nunique())
        baseline_invariance_records.append({
            "RepetitionSeed": repetition_seed,
            "Technique": technique,
            "Conditions": int(len(rows)),
            "KeyVariantsAcrossNoise": key_variants,
            "ScoreVariantsAcrossNoise": score_variants,
            "RankVariantsAcrossNoise": rank_variants,
            "Pass": bool(
                len(rows) == len(NOISE_LEVELS)
                and key_variants == 1
                and score_variants == 1
                and rank_variants == 1
            ),
        })

worker_baseline_invariance = pd.DataFrame(baseline_invariance_records)
baseline_invariance_failures = int((~worker_baseline_invariance["Pass"]).sum())
worker_qtf_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "ScoreSHA256",
    ].nunique()
)
worker_qtf_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints["Technique"].eq("QTF-Avg"),
        "RankSHA256",
    ].nunique()
)

zero_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].eq(0)
]
positive_audit = combined_condition_audit.loc[
    combined_condition_audit["NoisePercent"].gt(0)
]
noise_plan_hash_mismatches = int(
    combined_condition_audit["ExpectedFlipMaskSHA256"].ne(
        combined_condition_audit["ActualFlipMaskSHA256"]
    ).sum()
    + combined_condition_audit["ExpectedNoisyRawVerdictSHA256"].ne(
        combined_condition_audit["ActualNoisyRawVerdictSHA256"]
    ).sum()
    + combined_condition_audit["ExpectedNoisyModelVerdictSHA256"].ne(
        combined_condition_audit["ActualNoisyModelVerdictSHA256"]
    ).sum()
)

shared_equivalence = load_valid_shared_equivalence()
shared_equivalence_rows = 0 if shared_equivalence is None else len(shared_equivalence)

registry_sha256_after = sha256_file(REGISTRY_PATH)
current_source_rows_after = []
for row in frozen_source_manifest.itertuples(index=False):
    source_path = SOURCE_DIR / str(row.RelativePath)
    current_source_rows_after.append({
        "RelativePath": str(row.RelativePath),
        "SizeBytes": int(source_path.stat().st_size),
        "SHA256": sha256_file(source_path),
    })
source_root_sha256_after = source_root_hash(pd.DataFrame(current_source_rows_after))

master_write_guard_after = {
    str(path): path_guard_state(path)
    for path in MASTER_WRITE_GUARD_PATHS
}
master_write_guard_failures = sum(
    int(master_write_guard_before[path] != master_write_guard_after[path])
    for path in master_write_guard_before
)

validation_records = []
add_check(validation_records, "Worker assigned seeds", WORKER_REPETITION_SEEDS, sorted(worker_inventory["RepetitionSeed"].unique().tolist()), sorted(worker_inventory["RepetitionSeed"].unique().tolist()) == WORKER_REPETITION_SEEDS)
add_check(validation_records, "Worker conditions", EXPECTED_WORKER_CONDITIONS, len(worker_inventory), len(worker_inventory) == EXPECTED_WORKER_CONDITIONS)
add_check(validation_records, "Worker noise levels", NOISE_LEVELS, sorted(worker_inventory["NoisePercent"].unique().tolist()), sorted(worker_inventory["NoisePercent"].unique().tolist()) == NOISE_LEVELS)
add_check(validation_records, "Duplicate condition keys", 0, int(worker_inventory["ConditionKey"].duplicated(keep=False).sum()), not worker_inventory["ConditionKey"].duplicated(keep=False).any())
add_check(validation_records, "Files per condition", EXPECTED_FILES_PER_CONDITION, sorted(worker_inventory["Files"].unique().tolist()), worker_inventory["Files"].eq(EXPECTED_FILES_PER_CONDITION).all())
add_check(validation_records, "Worker raw files", EXPECTED_WORKER_RAW_FILES, worker_raw_files, worker_raw_files == EXPECTED_WORKER_RAW_FILES)
add_check(validation_records, "Worker ranking rows", EXPECTED_WORKER_RANKING_ROWS, int(worker_inventory["RankingRows"].sum()), int(worker_inventory["RankingRows"].sum()) == EXPECTED_WORKER_RANKING_ROWS)
add_check(validation_records, "Worker build-metric rows", EXPECTED_WORKER_BUILD_METRIC_ROWS, build_metric_rows, build_metric_rows == EXPECTED_WORKER_BUILD_METRIC_ROWS)
add_check(validation_records, "Worker project-run rows", EXPECTED_WORKER_PROJECT_RUN_ROWS, project_run_rows, project_run_rows == EXPECTED_WORKER_PROJECT_RUN_ROWS)
add_check(validation_records, "Worker model fits", EXPECTED_WORKER_MODEL_FITS, model_fit_rows, model_fit_rows == EXPECTED_WORKER_MODEL_FITS)
add_check(validation_records, "Worker condition-audit rows", EXPECTED_WORKER_CONDITIONS, len(combined_condition_audit), len(combined_condition_audit) == EXPECTED_WORKER_CONDITIONS)
add_check(validation_records, "Worker training-median rows", EXPECTED_WORKER_TRAINING_MEDIAN_ROWS, int(worker_inventory["TrainingMedianRows"].sum()), int(worker_inventory["TrainingMedianRows"].sum()) == EXPECTED_WORKER_TRAINING_MEDIAN_ROWS)
add_check(validation_records, "Zero-noise conditions", len(WORKER_REPETITION_SEEDS), len(zero_audit), len(zero_audit) == len(WORKER_REPETITION_SEEDS))
add_check(validation_records, "Zero-noise raw flips", 0, int(zero_audit["NumberFlipped"].sum()), int(zero_audit["NumberFlipped"].sum()) == 0)
add_check(validation_records, "Zero-noise model-label changes", 0, int(zero_audit["ModelLabelChanges"].sum()), int(zero_audit["ModelLabelChanges"].sum()) == 0)
add_check(validation_records, "Zero-noise dependent REC changes", 0, int(zero_audit["DependentRECChanges"].sum()), int(zero_audit["DependentRECChanges"].sum()) == 0)
add_check(validation_records, "Positive-noise raw-change violations", 0, int(positive_audit["NumberFlipped"].le(0).sum()), not positive_audit["NumberFlipped"].le(0).any())
add_check(validation_records, "Positive-noise model-change violations", 0, int(positive_audit["ModelLabelChanges"].le(0).sum()), not positive_audit["ModelLabelChanges"].le(0).any())
add_check(validation_records, "Positive-noise dependent-REC violations", 0, int(positive_audit["DependentRECChanges"].le(0).sum()), not positive_audit["DependentRECChanges"].le(0).any())
add_check(validation_records, "Independent REC changes", 0, int(combined_condition_audit["IndependentRECChanges"].sum()), int(combined_condition_audit["IndependentRECChanges"].sum()) == 0)
add_check(validation_records, "Independent reconstruction mismatches", 0, int(combined_condition_audit["IndependentReconstructionMismatches"].sum()), int(combined_condition_audit["IndependentReconstructionMismatches"].sum()) == 0)
add_check(validation_records, "Noise-plan hash mismatches", 0, noise_plan_hash_mismatches, noise_plan_hash_mismatches == 0)
add_check(validation_records, "Baseline invariance failures", 0, baseline_invariance_failures, baseline_invariance_failures == 0)
add_check(validation_records, "Worker QTF score variants", 1, worker_qtf_score_variants, worker_qtf_score_variants == 1)
add_check(validation_records, "Worker QTF rank variants", 1, worker_qtf_rank_variants, worker_qtf_rank_variants == 1)
add_check(validation_records, "Shared accelerated smoke-equivalence rows", 2, shared_equivalence_rows, shared_equivalence_rows == 2)
add_check(validation_records, "Accelerated clean REC mismatches", 0, accelerated_clean_mismatch_values, accelerated_clean_mismatch_values == 0)
add_check(validation_records, "Source root unchanged", EXPECTED_SOURCE_ROOT_SHA256, source_root_sha256_after, source_root_sha256_after == EXPECTED_SOURCE_ROOT_SHA256)
add_check(validation_records, "Completion registry unchanged", registry_sha256_before, registry_sha256_after, registry_sha256_after == registry_sha256_before)
add_check(validation_records, "Master Step 5A write-guard failures", 0, master_write_guard_failures, master_write_guard_failures == 0)
add_check(validation_records, "Registry Project 24 rows", 0, int(registry_project_numbers.eq(PROJECT_NUMBER).sum()), int(registry_project_numbers.eq(PROJECT_NUMBER).sum()) == 0)

worker_validation = pd.DataFrame(validation_records)
failed_validation = worker_validation.loc[~worker_validation["Pass"]]

print("\nWorker validation:")
display(worker_validation)

if not failed_validation.empty:
    print("\nFailed worker checks:")
    display(failed_validation)
    raise RuntimeError(
        f"PROJECT 24 STEP 5A WORKER {WORKER_TAG} VALIDATION FAILED."
    )

# --------------------------------------------------------------------------------------------------
# 10. FREEZE PRIVATE WORKER CHECKPOINT ONLY
# --------------------------------------------------------------------------------------------------

atomic_csv(WORKER_INVENTORY_PATH, worker_inventory)
atomic_csv(WORKER_RAW_MANIFEST_PATH, worker_raw_manifest)
atomic_csv(WORKER_BASELINE_INVARIANCE_PATH, worker_baseline_invariance)
atomic_csv(WORKER_COMBINED_AUDIT_PATH, combined_condition_audit)
atomic_csv(WORKER_VALIDATION_PATH, worker_validation)

worker_execution_seconds = time.perf_counter() - full_execution_started
completed_at_utc = datetime.now(timezone.utc).isoformat()

worker_output_paths = [
    WORKER_INVENTORY_PATH,
    WORKER_RAW_MANIFEST_PATH,
    WORKER_BASELINE_INVARIANCE_PATH,
    WORKER_COMBINED_AUDIT_PATH,
    WORKER_VALIDATION_PATH,
]
worker_output_manifest = [
    {
        "Path": str(path),
        "Bytes": int(path.stat().st_size),
        "SHA256": sha256_file(path),
    }
    for path in worker_output_paths
]

worker_report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "WorkerTag": WORKER_TAG,
    "WorkerNotebookName": WORKER_NOTEBOOK_NAME,
    "WorkerCodeRevision": WORKER_CODE_REVISION,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "RECImplementation": EXPECTED_REC_IMPLEMENTATION,
    "Step3ACodeRevision": EXPECTED_STEP3A_CODE_REVISION,
    "RuntimeVersions": actual_runtime_versions,
    "Status": WORKER_STATUS,
    "CompletedAtUTC": completed_at_utc,
    "AssignedSeeds": WORKER_REPETITION_SEEDS,
    "AssignedConditions": EXPECTED_WORKER_CONDITIONS,
    "CompletedConditions": len(worker_inventory),
    "NoiseLevels": NOISE_LEVELS,
    "MLFits": model_fit_rows,
    "RankingRows": int(worker_inventory["RankingRows"].sum()),
    "BuildMetricRows": build_metric_rows,
    "ProjectRunRows": project_run_rows,
    "ConditionAuditRows": len(combined_condition_audit),
    "TrainingMedianRows": int(worker_inventory["TrainingMedianRows"].sum()),
    "WorkerRawFiles": worker_raw_files,
    "WorkerRawBytes": worker_raw_bytes,
    "WorkerRawRootSHA256": worker_raw_root_sha256,
    "AcceleratedCleanRECMismatches": accelerated_clean_mismatch_values,
    "SharedSmokeEquivalencePath": str(ACCELERATED_EQUIVALENCE_PATH),
    "SharedSmokeEquivalenceSHA256": sha256_file(ACCELERATED_EQUIVALENCE_PATH),
    "BaselineInvarianceFailures": baseline_invariance_failures,
    "ValidationChecks": len(worker_validation),
    "FailedValidationChecks": len(failed_validation),
    "CompletedThisInvocation": completed_this_run,
    "SkippedValidatedConditions": skipped_valid,
    "WorkerExecutionSecondsThisInvocation": float(worker_execution_seconds),
    "WorkerOutputManifest": worker_output_manifest,
    "SourceRootSHA256": source_root_sha256_after,
    "RegistrySHA256": registry_sha256_after,
    "RegistryModified": False,
    "MasterStep5AWriteGuardFailures": master_write_guard_failures,
    "Projects1To23Modified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "ResumeSafe": True,
    "MasterFinalizationRequired": True,
}
atomic_json(WORKER_REPORT_PATH, worker_report)

worker_checkpoint = {
    **worker_report,
    "CheckpointVersion": 1,
    "WorkerReportPath": str(WORKER_REPORT_PATH),
    "WorkerReportSHA256": sha256_file(WORKER_REPORT_PATH),
    "WorkerShardComplete": True,
    "OfficialStep5AComplete": False,
    "NextRequiredStep": "WAIT_FOR_ALL_SIX_WORKERS_THEN_MASTER_STEP5A_FINALIZATION",
}
atomic_json(WORKER_CHECKPOINT_PATH, worker_checkpoint)

worker_status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "WorkerTag": WORKER_TAG,
    "WorkerCodeRevision": WORKER_CODE_REVISION,
    "AcceleratedEngineVersion": ACCELERATED_ENGINE_VERSION,
    "Status": WORKER_STATUS,
    "AssignedSeeds": WORKER_REPETITION_SEEDS,
    "AssignedConditions": EXPECTED_WORKER_CONDITIONS,
    "CompletedConditions": len(worker_inventory),
    "WorkerRawFiles": worker_raw_files,
    "WorkerRawRootSHA256": worker_raw_root_sha256,
    "Checkpoint": str(WORKER_CHECKPOINT_PATH),
    "CheckpointSHA256": sha256_file(WORKER_CHECKPOINT_PATH),
    "OfficialStep5AComplete": False,
}
atomic_json(WORKER_STATUS_PATH, worker_status_payload)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "WorkerTag": WORKER_TAG,
        "Status": WORKER_STATUS,
        "UpdatedAtUTC": completed_at_utc,
        "CompletedConditions": EXPECTED_WORKER_CONDITIONS,
        "ExpectedConditions": EXPECTED_WORKER_CONDITIONS,
        "AssignedSeeds": WORKER_REPETITION_SEEDS,
        "CheckpointSHA256": sha256_file(WORKER_CHECKPOINT_PATH),
        "ResumeSafe": True,
        "OfficialStep5AComplete": False,
    },
)

checkpoint_readback = load_json(WORKER_CHECKPOINT_PATH)
if checkpoint_readback.get("Status") != WORKER_STATUS:
    raise RuntimeError("Worker checkpoint readback failed.")
if sha256_file(REGISTRY_PATH) != registry_sha256_before:
    raise RuntimeError("Completion registry changed during worker finalisation.")

final_master_guard_after = {
    str(path): path_guard_state(path)
    for path in MASTER_WRITE_GUARD_PATHS
}
if any(
    master_write_guard_before[path] != final_master_guard_after[path]
    for path in master_write_guard_before
):
    raise RuntimeError("A master Step 5A output changed during worker finalisation.")

print("\n" + "=" * 136)
print(f"=== PROJECT 24 STEP 5A WORKER {WORKER_TAG} RESULT ===")
print("=" * 136)
print("Project:", PROJECT_NAME)
print("Worker notebook:", WORKER_NOTEBOOK_NAME)
print("Worker code revision:", WORKER_CODE_REVISION)
print("Accelerated engine:", ACCELERATED_ENGINE_VERSION)
print("Assigned seeds:", WORKER_REPETITION_SEEDS)
print("Conditions:", len(worker_inventory), "/", EXPECTED_WORKER_CONDITIONS)
print("ML fits:", model_fit_rows, "/", EXPECTED_WORKER_MODEL_FITS)
print("Ranking rows:", int(worker_inventory["RankingRows"].sum()))
print("Build-metric rows:", build_metric_rows)
print("Project-run rows:", project_run_rows)
print("Condition-audit rows:", len(combined_condition_audit))
print("Training-median rows:", int(worker_inventory["TrainingMedianRows"].sum()))
print("Worker raw files:", worker_raw_files)
print("Worker raw bytes:", worker_raw_bytes)
print("Worker raw root SHA-256:", worker_raw_root_sha256)
print("Baseline invariance failures:", baseline_invariance_failures)
print("Master Step 5A write-guard failures:", master_write_guard_failures)
print("Validation checks:", len(worker_validation))
print("Failed checks:", len(failed_validation))
print("Completed this invocation:", completed_this_run)
print("Skipped validated conditions:", skipped_valid)
print("Runtime seconds this invocation:", round(worker_execution_seconds, 2))
print("Worker checkpoint:", WORKER_CHECKPOINT_PATH)
print("Worker checkpoint SHA-256:", sha256_file(WORKER_CHECKPOINT_PATH))
print("Official master Step 5A complete:", False)
print("STATUS:", WORKER_STATUS)
print("=" * 136)


=== PROJECT 24 STEP 5A PARALLEL WORKER: SEEDS 01-05 ===
Worker notebook: Thesis_p24_seed16-20
Assigned repetition seeds: [16, 17, 18, 19, 20]
Assigned conditions: 45
Mounting the shared thesis Google Drive.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Worker runtime versions match frozen Step 4A exactly.

Loading frozen Project 24 cohorts and contracts.
Converting the fixed predictor cohorts to one numeric matrix.
Precomputing the vectorized verdict-dependent REC engine.
Accelerated REC engine clean-equivalence mismatches: 0
Accelerated REC groups / model rows / entities: 2641 / 224550 / 14725
Worker shard frozen: [16, 17, 18, 19, 20]
Worker conditions: 45
Master Step 5A write guard armed for 12 paths.

Scanning existing condition checkpoints...
Valid completed conditions: 0
Incomplete/invalid condition directories: 0
Pending assigned conditions: 45

Loading deterministic RNG stream for seed 16.

------

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_00__seed_16
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 1777 | condition seconds: 128.63

--------------------------------------------------------------------------------------------------------------
[worker 2/45 | global 137/270] Running noise_05__seed_16
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_05__seed_16
  Raw flips: 202366 | model-label changes: 10263 | dependent REC changes: 1756635
  Training failures: 11870 | condition seconds: 266.73

--------------------------------------------------------------------------------------------------------------
[worker 3/45 | global 138/270] Running noise_10__seed_16
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


...

  Fitting: NaiveBayes
  Completed: noise_40__seed_20
  Raw flips: 1616779 | model-label changes: 82322 | dependent REC changes: 2509178
  Training failures: 82705 | condition seconds: 244.18

--------------------------------------------------------------------------------------------------------------
[worker 45/45 | global 180/270] Running noise_50__seed_20
--------------------------------------------------------------------------------------------------------------
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_20
  Raw flips: 2020562 | model-label changes: 102951 | dependent REC changes: 2574571
  Training failures: 102978 | condition seconds: 260.55

Validating all 45 assigned conditions for seed_16_20.

Worker validation:


,Check,Expected,Actual,Pass
0,Worker assigned seeds,"[16, 17, 18, 19, 20]","[16, 17, 18, 19, 20]",True
1,Worker conditions,45,45,True
2,Worker noise levels,"[0, 5, 10, 15, 20, 25, 30, 40, 50]","[0, 5, 10, 15, 20, 25, 30, 40, 50]",True
3,Duplicate condition keys,0,0,True
4,Files per condition,8,[8],True
5,Worker raw files,360,360,True
6,Worker ranking rows,5939010,5939010,True
7,Worker build-metric rows,5355,5355,True
8,Worker project-run rows,315,315,True
9,Worker model fits,180,180,True



=== PROJECT 24 STEP 5A WORKER seed_16_20 RESULT ===
Project: SonarSource@sonarqube
Worker notebook: Thesis_p24_seed16-20
Worker code revision: P24_SEEDS_16_20_V1_FROZEN_SCHEMA_SMOKE_EQUIVALENT
Accelerated engine: PROJECT_24_FAST_DEPENDENT_REC_V1_SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE
Assigned seeds: [16, 17, 18, 19, 20]
Conditions: 45 / 45
ML fits: 180 / 180
Ranking rows: 5939010
Build-metric rows: 5355
Project-run rows: 315
Condition-audit rows: 45
Training-median rows: 6795
Worker raw files: 360
Worker raw bytes: 87224279
Worker raw root SHA-256: 910ef203274d1ea744ad92ef1ec7802777d38be713333b21f5191fa69d4c364a
Baseline invariance failures: 0
Master Step 5A write-guard failures: 0
Validation checks: 31
Failed checks: 0
Completed this invocation: 45
Skipped validated conditions: 0
Runtime seconds this invocation: 10913.34
Worker checkpoint: /content/drive/MyDrive/Thesis_Experiment/Notes/project_24_step5a_worker_seed_16_20_checkpoint.json
Worker checkpoint SHA-256: 31